# 02 SDXL Base 1.0 の中身を見る (all-in-one 統合版) — text → embedding → cross-attention → 拡散過程 → VAE

このノートブックは、[01ノートブック (01_sdxl_base_intro.ipynb)](01_sdxl_base_intro.ipynb) で構造をなめた [SDXL Base 1.0](https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0) に対して、**「prompt から画像になるまでの内部処理を、実物のコードの中身を開いて見る」こと**を目的にしています。3 年前 (2023) の SD1.5 デモで cross-attention の可視化や denoising trajectory を見せていた内容を、SDXL Base で組み直したものです。

> **note**: 実行時間の目安 (cache hit・2 回目以降、1 回流し切り)。GPU が **fp16 か fp32 か**で大差が出ます (MPS は SDXL の fp16 NaN 問題を避けて fp32 を使うので重い):
>
> | 環境 | device / dtype | 実行時間 |
> |---|---|---:|
> | Mac (M4 Max 64GB) | MPS / fp32 | ~12 分 |
> | Mac Studio (M1 Ultra 128GB) | MPS / fp32 | ~5.5 分 |
> | Windows (RTX 5080) | CUDA / fp16 | ~2.5 分 (154 s) |
> | Colab (**L4**) | CUDA / fp16 | ~3 分 |
>
> **Colab で動かすときは L4 を選んでください** — GPU RAM を ~15.5 GB 使うため T4 (15 GB) では不足します。ローカルは **メモリ 64GB 以上**を推奨。初回のみ SDXL Base の DL が入る（**fp16 の Colab/Windows は ~7GB、fp32 の Mac は ~14GB**。回線次第で数分〜、2 回目以降は `~/.cache/huggingface/` から数秒で load）。ファイルサイズ ~24 MB。

## 0. はじめに

### このノートブックでやること

[01ノートブック](01_sdxl_base_intro.ipynb) と同じ「魔法使い」prompt + seed を使い、**今度はパイプラインを `pipe(...)` で 1 発呼ばずに、各段を手動でほどいて中身を見ます**。

1. [環境確認](#env) — device (CPU / MPS / CUDA) と dtype
2. [モデル読み込み](#load) — `StableDiffusionXLPipeline` ([01ノートブック](01_sdxl_base_intro.ipynb) と同じ load 手順)
3. [prompt と生成パラメータ](#prompt) — [01ノートブック](01_sdxl_base_intro.ipynb) と同じ prompt / seed / 1024×1024 / 30 steps
4. [tokenize → text encode → encode_prompt](#tokenize) — 2 系統の text encoder の出力 (768 + 1280 = 2048) と pooled (1280) を分解
5. [prompt embedding の地形図](#embed) — 17 prompt を encode → pooled を PCA + t-SNE で 2D 散布
6. [手動 scheduler ループで denoising trajectory](#trajectory) — 30 step を手で回しながら中間 latent を VAE decode、「ノイズから絵が立ち上がる」過程
7. [cross-attention probe](#attn) — `RecordingAttnProcessor` を当てて 5 step の attention weight を capture。§7.2 (UNet 構造) → §7.3-7.5 (準備) → §7.6 (1 head の raw) → §7.7 (.t0 head PCA-RGB) → §7.8 (全 t-block pool PCA-RGB) → §7.9 (profile-based head finding) → §7.10 (self-attn) → §7.11 (画像→単語 view)
8. [guidance / negative prompt](#cfg) — guidance_scale = {0, 3, 7.5, 12} sweep + negative prompt on/off
9. [VAE の roundtrip](#vae) — §6 の final image を再 encode → decode して圧縮損失を見る
10. [まとめ](#summary) — 中身を 5 ステップで再構成

### 生成画像の保存

生成図は notebook 内表示に加えて `outputs/02_sdxl_base_inside/` ディレクトリに PNG として書き出します (`sec5_*`, `sec6_*`, `sec7_*`, `sec8_*`, `sec9_*` で計十数枚)。

### 前提

- [01ノートブック](01_sdxl_base_intro.ipynb) (SDXL Base intro) を一度通したことを想定。Pipeline の構成・disk 容量・SD1.5 との差分は [01ノートブック](01_sdxl_base_intro.ipynb) を参照
- 1024×1024 / 30 steps / **MPS では fp32** (SDXL は MPS + fp16 で NaN が出る既知問題のため)。M4 Max で 1 回流し切るのに ~12 分程度を見込む (詳細は冒頭 note)
- 共通環境 `~/.venvs/aidemo2026` (`requirements.txt`) で動くことを目指す (scikit-learn, matplotlib, numpy, pandas を使用)

### このノート単体での読み筋

各 § は独立に読めるように作っていますが、依存関係は以下:

- §4 (encode_prompt 分解) → §5, §6, §7 すべて (prompt_embeds と pooled を作る出発点)
- §6 (手動 trajectory) → §7 で同じ手動 loop に attention hook を当てる、§9 で final image を使う
- §7, §8 は §6 と独立


## 0. 環境セットアップ（3環境 自動切替: Colabだけ pip / path も自動）


In [ ]:
# 3環境(Mac/Win/Colab)を同一ファイルで動かすための判定。Colabのみ pip（Mac/Winはenvに在るのでskip）。
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q "transformers==5.9.0" "accelerate==1.13.0"
    print("Colab: pip done")
else:
    print("local(Mac/Win): pip skip（env利用）")

<a id="env"></a>

---
## 1. 環境確認

[01ノートブック (01_sdxl_base_intro.ipynb)](01_sdxl_base_intro.ipynb) と同じ。device と dtype を選びます。

**dtype の注意**:

- **SDXL VAE は fp16 で NaN を出す既知問題** (CUDA / MPS 共通) があり、`madebyollin/sdxl-vae-fp16-fix` のような fp16-safe な再学習版 VAE が公開されているほど。本ノートでは公式 VAE をそのまま使うので、VAE は fp32 が必要
- **MPS では加えて UNet も fp16 / bf16 で不安定** (本環境で観測、詳細は [docs/01b_sdxl_base_mps_fp16_probe.ipynb](../docs/01b_sdxl_base_mps_fp16_probe.ipynb))

以上を踏まえ、本ノート (MPS 想定) では UNet / VAE どちらも安全側に倒して fp32 を選びます (CUDA / CPU はそれぞれ fp16 / fp32)。


In [ ]:
import sys
import logging
import torch
import diffusers
import transformers

logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("Diffusers   :", diffusers.__version__)
print("Transformers:", transformers.__version__)

if torch.cuda.is_available():
    device = "cuda"
    dtype = torch.float16
elif torch.backends.mps.is_available():
    device = "mps"
    dtype = torch.float32
else:
    device = "cpu"
    dtype = torch.float32

print("device      :", device)
print("dtype       :", dtype)


### matplotlib の日本語フォント設定 (Tier 1: CJK)

§5 以降の図で日本語タイトル / ラベルを文字化けさせないため、`rcParams` に CJK 対応 font を優先順で設定します。matplotlib の default `DejaVu Sans` は CJK glyph を持たないので □□ になるためです。

3 環境で動くよう fallback chain にしてあります:

- **macOS**: `Hiragino Sans` (標準搭載)
- **Windows**: `Yu Gothic` / `Meiryo` (標準搭載)
- **Colab / Linux**: `fonts-ipafont-gothic` (IPAGothic) を apt で導入し `fontManager` に登録 (無いと Colab で日本語が豆腐になる)
- いずれもなければ `DejaVu Sans` に戻す (= ASCII のみは正常表示、CJK は □□)

この図で描くのは日本語タイトルと英語プロンプトのトークンラベルのみなので CJK (Tier 1) で十分です (アラビア語など RTL 対応の Tier 2 は不要)。

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# Tier 1: CJK フォント（日本語）— 図のタイトル/凡例に日本語を使うため。
# matplotlib 既定 DejaVu は CJK glyph を持たない。★ Colab は CJK フォントが無いので apt 導入が必須
# （無いと Colab で日本語タイトルが豆腐になる）。Mac/Win は system フォントで足りる。
import sys as _sys
if "google.colab" in _sys.modules:
    import subprocess as _sp, glob as _gl
    _sp.run(["apt-get", "-qq", "-y", "install", "fonts-ipafont-gothic", "fonts-wqy-zenhei"], check=False)
    for _p in ("/usr/share/fonts/**/ipag*.ttf", "/usr/share/fonts/**/wqy-zenhei*.tt?"):
        for _f in _gl.glob(_p, recursive=True):
            try: fm.fontManager.addfont(_f)
            except Exception: pass
available = {f.name for f in fm.fontManager.ttflist}
CJK_FONT_CANDIDATES = [
    "IPAGothic", "Hiragino Sans", "Yu Gothic", "Meiryo", "Noto Sans CJK JP", "Noto Sans JP",  # 日本語
    "PingFang SC", "Microsoft YaHei", "WenQuanYi Zen Hei", "Noto Sans CJK SC",                 # 簡体字
    "Arial Unicode MS",
]
preferred = [n for n in CJK_FONT_CANDIDATES if n in available]
plt.rcParams["font.family"] = preferred + ["DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False  # マイナス符号の文字化け回避

print("matplotlib   :", matplotlib.__version__)
print("CJK font 候補 (本環境で利用可):", preferred or "(なし、ASCII のみ正常表示)")

<a id="load"></a>

---
## 2. モデル読み込み

[01ノートブック §2](01_sdxl_base_intro.ipynb#load) と同じ手順で SDXL Base 1.0 を読み込みます。cache hit していれば数秒、初回は約 7-14 GB の download が走るので十数分〜1 時間かかります (詳細は [01ノートブック §4](01_sdxl_base_intro.ipynb#peek) の cache 表参照)。


In [ ]:
MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"
print("model_id:", MODEL_ID)


In [ ]:
import time

from diffusers import StableDiffusionXLPipeline

# fp16/bf16 → variant="fp16" (約 7 GB)、fp32 → variant=None (約 14 GB)
variant = "fp16" if dtype in (torch.float16, torch.bfloat16) else None

print(f"pipeline 読み込み中... variant={variant!r}")
t_load = time.perf_counter()
pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    use_safetensors=True,
    variant=variant,
)
pipe = pipe.to(device)
load_elapsed = time.perf_counter() - t_load

if device == "mps":
    pipe.enable_attention_slicing()
    print("[mps] enable_attention_slicing() 有効化")

print(f"load done in {load_elapsed:.2f} s")
print("pipeline クラス:", type(pipe).__name__)


<a id="prompt"></a>

---
## 3. prompt と生成パラメータ

[01ノートブック](01_sdxl_base_intro.ipynb) と同じ「魔法使い」prompt + seed を使います。**同じ prompt で [01ノートブック](01_sdxl_base_intro.ipynb) と直接比較できる**ようにするためです。[01ノートブック §3](01_sdxl_base_intro.ipynb#gen) と同じ画像を `pipe(...)` 1 発で作るのではなく、本ノートブック§4 以降で手動でほどいていくのが 02 の主役です。


In [ ]:
width = 1024
height = 1024
num_inference_steps = 30
guidance_scale = 7.5

prompt = (
    "a young witch with long flowing hair holding a glowing magic staff, "
    "in an enchanted forest with bioluminescent mushrooms, "
    "fantasy art, masterpiece, detailed"
)
negative_prompt = (
    "blurry, low quality, distorted, bad anatomy, extra limbs, "
    "watermark, signature"
)
seed = 123

print("prompt          :", prompt)
print("negative_prompt :", negative_prompt)
print("seed            :", seed)
print("size            :", f"{width} x {height}")
print("steps           :", num_inference_steps)
print("guidance_scale  :", guidance_scale)


In [ ]:
from pathlib import Path

# notebook は lecture/ から実行される前提 → 親ディレクトリの outputs/ を使う
NB_OUT_DIR = (Path("outputs/02_sdxl_base_inside") if IN_COLAB else Path("../outputs/02_sdxl_base_inside"))
NB_OUT_DIR.mkdir(parents=True, exist_ok=True)
print("output dir :", NB_OUT_DIR)


<a id="tokenize"></a>

---
## 4. tokenize → text encode → encode_prompt の中身を覗く

SDXL Base の text 条件付けは、**2 つの tokenizer + 2 つの text encoder の出力を concat して 2048 次元の embedding にする**仕組みでした ([01ノートブック §5](01_sdxl_base_intro.ipynb#inside))。[01ノートブック](01_sdxl_base_intro.ipynb) では「2 つあるね」までだったのを、02 では tensor の shape まで自分の目で確かめます。

最終的に作りたいのは UNet が受け取る:

- `prompt_embeds` : `(1, 77, 2048)`  = `[CLIP-L (1,77,768)  ‖  OpenCLIP-G (1,77,1280)]`
- `pooled_prompt_embeds` : `(1, 1280)`  = OpenCLIP-G の pooled output (sentence-level の大局条件)

の 2 つです。これは `pipe.encode_prompt(...)` 1 行で出ますが、ここでは tokenizer / text_encoder を別々に呼んで concat した結果と pipeline 内蔵関数の出力が一致することを検算します。


### 4.1 tokenizer × 2 — 同じ id 列を 2 つの encoder に流す

[01ノートブック §6](01_sdxl_base_intro.ipynb#tokenizer) で確認したとおり、SDXL Base の `tokenizer` と `tokenizer_2` は **同じ CLIP の vocab を共有**しています。「tokenizer が 2 つある」は token 化レイヤの違いではなく、**同じ token id 列を 2 つの異なる text encoder に流すことで 2 系統の embedding が得られる**ことが本質でした。

なお SDXL が 2 系統を concat する狙いは、**訓練 dataset / scale が違う 2 つの encoder の相補的 strengths を合体させる**こと。OpenAI CLIP-L (2021、768d) は語彙レベルの細かい区別に強く、LAION OpenCLIP-bigG (2022、1280d) は大規模 web 由来の concept 表現に強い。両者を最終次元方向で結合して 2048d の text 条件を作り、UNet の cross-attention に渡す。


In [ ]:
tokens_L = pipe.tokenizer(
    prompt, padding="max_length", max_length=pipe.tokenizer.model_max_length,
    truncation=True, return_tensors="pt",
)
tokens_G = pipe.tokenizer_2(
    prompt, padding="max_length", max_length=pipe.tokenizer_2.model_max_length,
    truncation=True, return_tensors="pt",
)

print("CLIP-L     token ids shape :", tuple(tokens_L.input_ids.shape))
print("OpenCLIP-G token ids shape :", tuple(tokens_G.input_ids.shape))
print()
ids_L = tokens_L.input_ids[0].tolist()
ids_G = tokens_G.input_ids[0].tolist()
print("token 列の長さ (= max_length, 77 で padding 済):")
print("  CLIP-L     :", len(ids_L))
print("  OpenCLIP-G :", len(ids_G))
print()
print("token id 全 77 要素 (padding 部分も含む):")
print("  CLIP-L     :", ids_L)
print("  OpenCLIP-G :", ids_G)
print("  same?      :", ids_L == ids_G)
print()
# 何 token 目で <|endoftext|> (id=49407) が出るか
eos_id = pipe.tokenizer.eos_token_id
n_real_L = ids_L.index(eos_id) + 1 if eos_id in ids_L else len(ids_L)
print(f"実際の prompt 部分 (BOS + tokens + EOS) : {n_real_L} tokens / 77")


### 4.2 text_encoder × 2 を個別に走らせる

SDXL の慣習として、`text_encoder.hidden_states[-2]` (penultimate layer) を使います。`output_hidden_states=True` で全層の hidden state を取り出します。


In [ ]:
tokens_L = {k: v.to(device) for k, v in tokens_L.items()}
tokens_G = {k: v.to(device) for k, v in tokens_G.items()}

with torch.no_grad():
    out_L = pipe.text_encoder(tokens_L["input_ids"], output_hidden_states=True)
    out_G = pipe.text_encoder_2(tokens_G["input_ids"], output_hidden_states=True)

emb_L = out_L.hidden_states[-2]  # SDXL は penultimate layer を使う
emb_G = out_G.hidden_states[-2]
pooled_G = out_G[0]              # CLIPTextModelWithProjection の projection output (text_embeds)

print("CLIP-L     hidden_states[-2] shape :", tuple(emb_L.shape))
print("OpenCLIP-G hidden_states[-2] shape :", tuple(emb_G.shape))
print("OpenCLIP-G pooled (text_embeds)    :", tuple(pooled_G.shape))
print()
print("CLIP-L     hidden_size :", pipe.text_encoder.config.hidden_size,
      "  layers :", pipe.text_encoder.config.num_hidden_layers)
print("OpenCLIP-G hidden_size :", pipe.text_encoder_2.config.hidden_size,
      "  layers :", pipe.text_encoder_2.config.num_hidden_layers)


In [ ]:
emb_L

In [ ]:
emb_G

In [ ]:
pooled_G

### 4.3 concat — 768 + 1280 = 2048 を自分の手で組む


In [ ]:
prompt_embeds_manual = torch.cat([emb_L, emb_G], dim=-1)
print("manual concat shape :", tuple(prompt_embeds_manual.shape))  # (1, 77, 2048)

# UNet が要求する cross_attention_dim と一致するか
print("unet.cross_attention_dim :", pipe.unet.config.cross_attention_dim)
assert prompt_embeds_manual.shape[-1] == pipe.unet.config.cross_attention_dim


In [ ]:
prompt_embeds_manual

### 4.4 `pipe.encode_prompt(...)` と一致するか検算

pipeline 内蔵の `encode_prompt()` は、上の concat + negative prompt 側の同じ処理 + CFG (Classifier-Free Guidance = 条件付き / 無条件の 2 本立てで生成する仕組み) batch 化を一発でやってくれる関数です。手動 concat と数値一致するはずです。


In [ ]:
do_cfg = guidance_scale > 1.0
with torch.no_grad():
    pe, npe, pooled, npooled = pipe.encode_prompt(
        prompt=prompt,
        negative_prompt=negative_prompt or None,
        device=torch.device(device),
        num_images_per_prompt=1,
        do_classifier_free_guidance=do_cfg,
    )

print("pipe.encode_prompt() 戻り値:")
print(f"  prompt_embeds          shape : {tuple(pe.shape)}")
print(f"  negative_prompt_embeds shape : {tuple(npe.shape) if npe is not None else None}")
print(f"  pooled                 shape : {tuple(pooled.shape)}")
print(f"  negative_pooled        shape : {tuple(npooled.shape) if npooled is not None else None}")
print()
diff_emb = (pe - prompt_embeds_manual).abs().max().item()
diff_pooled = (pooled - pooled_G).abs().max().item()
print(f"manual concat と pipeline 出力の差分 (positive 側):")
print(f"  prompt_embeds max abs diff : {diff_emb:.2e}")
print(f"  pooled        max abs diff : {diff_pooled:.2e}")


**まとめ — encode_prompt が中で何をしているか**

```
prompt (str)
 ├─ tokenizer   (CLIP-L vocab, 77 tokens)            → ids_L   (1, 77)
 │   └─ text_encoder   (CLIP ViT-L/14, 12 layers)    → hidden_states[-2]  (1, 77, 768)
 ├─ tokenizer_2 (同じ vocab, 77 tokens)              → ids_G   (1, 77) = ids_L
 │   └─ text_encoder_2 (OpenCLIP ViT-bigG/14, 32 l.) → hidden_states[-2]  (1, 77, 1280)
 │                                                   → projection (pooled)  (1, 1280)
 └─ concat([CLIP-L 768, OpenCLIP-G 1280], dim=-1)    → prompt_embeds        (1, 77, 2048)
                                                       pooled_prompt_embeds (1, 1280)

(do_classifier_free_guidance=True なら、negative_prompt にも同じ処理を流し、
 UNet 呼び出し時に [negative, positive] を batch=2 で concat する)
```

`(1, 77, 2048)` の `prompt_embeds` が UNet の cross-attention の Key/Value (テキスト側) として渡り、`(1, 1280)` の `pooled` は SDXL の `addition_embed_type="text_time"` 経由で time embedding に注入されて画像全体の大局条件として効きます ([01ノートブック §5](01_sdxl_base_intro.ipynb#inside) UNet の表参照)。


<a id="embed"></a>

---
## 5. prompt embedding の地形図 (PCA + t-SNE + cosine heatmap)

§4 で「prompt → (1, 77, 2048) + (1, 1280) の数値ベクトル」になることを確認しました。これを「複数の prompt について並べると、意味的に近い prompt は近くに固まるか?」を散布図と heatmap で見ます。本ノートでは **§3 で実際に使った prompt / negative_prompt 自身もバンクに加える** ことで、生成 prompt 自体が embedding 空間でどう振る舞うかも合わせて見せます (§5.1, §5.2)。

バンク構成 (合計 22 entries):

- **既存 17 prompt**: **5 クラスタ × 3-4 entries** の構成 — `魔法使い系` / `動物` / `食べ物` / `乗り物` / `風景・画風`。§3 と同系統の魔法使いクラスタを軸に、意図的に **意味的に遠い** 別クラスタを並べて、cluster 構造が embedding 空間で立つかを見る (§5.2)
- **§3 の prompt を 3 節に分割**: 長い prompt は節 (clause) に切って他の bank entry と長さを揃える (§5.2 で詳しく)
- **§3 の negative_prompt を 2 節に分割**: 同様

**単語と文を同じ意味空間で比較** するため、両方を caption-style テンプレートで揃えて encode します:

- **文** (PROMPT_BANK 各 entry): `"a photo of {text}"` で wrap
- **個別単語** (PROMPT_BANK から抽出): `"a photo of a {word}"` で wrap

text encoder からはどちらも 77 token 分の hidden states 列が出てくるので、**1 本の代表ベクトルに集約** するために **pooled (1, 1280)** (OpenCLIP-G の文全体表現、SDXL pipeline が `added_cond_kwargs["text_embeds"]` として正式に UNet へ渡す vector) を取り出します。文と単語の pooled を同じ空間に並べて、2D 可視化 (PCA + t-SNE) と **pairwise cosine 距離 heatmap** (§5.4) で「文 vs 単語」「文 vs 部分節」「単語 vs 単語」の距離関係を一度に見ます (§5.3)。

> **注意**: CLIP text encoder の出力は実は 2 系統あります — (1) **token ごとの hidden state** ((77, 2048)、UNet の cross-attention でそのまま参照される) と、(2) **全体を 1 本に集約した pooled** ((1, 1280)、SDXL では `text_embeds` として補助情報に使われる)。本節 §5 は「単語」と「文」を **同じ単一ベクトル空間** で並べるのが目的なので、上のテンプレ trick で単語側にも pooled を取らせています。ただし **実際の画像生成では、単語に pool は使われません** — (1) の token 列として cross-attention に入るのが本来の姿で、それは §7 の cross-attention probe で見ます。



### 5.1 §3 で実際に使った prompt / negative_prompt の tokenize を覗く

§4 で tokenize の中身を CLIP-L / OpenCLIP-G の両 tokenizer で見たが、ここでは **§3 で画像生成に使った実 prompt + negative_prompt** を例に、token id と **decoded piece (= BPE 分割された実テキスト断片)** を改めて表示する (§4 と内容は重複するが、本 §5 だけ読んでも完結するため再掲)。

これは §5.2 で「**prompt 自身を embedding バンクに加える**」前の準備。decoded piece を見ることで「単語が BPE でどう切られるか」「`witch` のような頻出語が 1 token か」「`bioluminescent` のような長い語が複数 token に分割されるか」を直感的に把握できる。


In [ ]:
# tokenize + decoded piece 表示 helper (CLIP-L tokenizer 使用、tokenizer_2 と同 vocab)
def tokenize_and_show(text: str, max_show: int = 25):
    enc = pipe.tokenizer(text, add_special_tokens=True)
    ids = enc.input_ids
    pieces = pipe.tokenizer.convert_ids_to_tokens(ids)
    print(f"  {len(ids)} tokens (BOS/EOS 含む、padding なし):")
    for i, (tid, piece) in enumerate(zip(ids[:max_show], pieces[:max_show])):
        marker = ""
        if piece == "<|startoftext|>":
            marker = "  ← BOS"
        elif piece == "<|endoftext|>":
            marker = "  ← EOS"
        print(f"    [{i:2d}] id={tid:5d}  piece={piece!r:24s}{marker}")
    if len(ids) > max_show:
        print(f"    ... ({len(ids) - max_show} more tokens)")

print("=== §3 の prompt ===")
print(f"  {prompt!r}\n")
tokenize_and_show(prompt)
print()
print("=== §3 の negative_prompt ===")
print(f"  {negative_prompt!r}\n")
tokenize_and_show(negative_prompt)


In [ ]:
# 既存 17 prompt — 5 クラスタ × 3-4 entries の構成。
# §3 と同系統の魔法使いクラスタを軸に、意図的に「意味的に遠い」別クラスタを
# 並べて、cluster 構造が embedding 空間で立つかを見る (3年前の SD1.5 デモと同じ意図)。
ORIGINAL_BANK = [
    # cluster A: 魔法使い+ファンタジー人物 (4) — §3 と同系統、内部に witch/wizard 性差ペアを含む
    "a young witch with magic staff in enchanted forest",
    "a young wizard with magic staff in enchanted forest",
    "an old wizard with long beard in a tower",
    "a knight with sword in a castle",
    # cluster B: 日常生物 (3)
    "a cat sitting in a garden",
    "a small dog playing on a beach",
    "a parrot perched on a branch",
    # cluster C: 食べ物・飲み物 (3)
    "a slice of pizza on a plate",
    "a bowl of ramen with chopsticks",
    "a cup of coffee on a wooden table",
    # cluster D: 乗り物 (3)
    "a red sports car on a highway",
    "a bicycle in a city street",
    "a sailboat on a calm sea",
    # cluster E: 風景+画風 (4) — 同じ "forest" を画風違いで対比するペアを含む
    "a watercolor painting of a forest",
    "a realistic photo of a forest",
    "a fantasy painting of a dragon",
    "an anime style portrait of a girl",
]

# cluster の境界 index (ORIGINAL_BANK 内の累積)。§5.4 heatmap で白線として描く。
CLUSTER_BOUNDARIES = [4, 7, 10, 13, 17]  # A: [0,4), B: [4,7), C: [7,10), D: [10,13), E: [13,17)
CLUSTER_NAMES = ["A: magic", "B: animal", "C: food", "D: vehicle", "E: scene/style"]

# §3 の prompt を節 (clause) に分割 — カンマで切って、短い style modifier 3 つは 1 節にまとめる
# (他の bank 例が 5-10 token なのに合わせる「長さ揃え」の操作)
# 注: item 2 は元 "in an enchanted forest..." だったが、§5.3 で sentence 側も
# "a photo of {p}" テンプレで wrap するため、頭の "in " を落として "an enchanted forest..." にした
# ("a photo of in ..." を回避)
PROMPT_CLAUSES = [
    "a young witch with long flowing hair holding a glowing magic staff",  # 主役の描写
    "an enchanted forest with bioluminescent mushrooms",                     # 場所
    "fantasy art, masterpiece, detailed",                                    # style 修飾 3 つ
]

# §3 の negative_prompt も節分割 (3+1=4 個の修飾語をカンマで 2 節に)
NEGATIVE_CLAUSES = [
    "blurry, low quality, distorted",
    "bad anatomy, extra limbs, watermark, signature",
]

PROMPT_BANK = ORIGINAL_BANK + PROMPT_CLAUSES + NEGATIVE_CLAUSES

# === テンプレ統一 (§5.3 で文と単語を同じ caption-style 構文に揃える) ===
# 単語側 "a photo of a {w}" だけだと「a photo of 方向のバイアス」が単語塊全体に乗り、
# scatter で「文クラスタ vs 単語クラスタ」の 2 大島構造が支配してしまう。
# 文側も同じ prefix "a photo of " でくるむと、共通 bias が cosine 距離から自然に
# 抜け、content 差が前面に出やすくなる (CLIP は caption 学習なので caption-style にも整合)。
SENTENCE_TEMPLATE = "a photo of {p}"
WORD_TEMPLATE     = "a photo of a {w}"

print(f"PROMPT_BANK: {len(PROMPT_BANK)} entries  "
      f"(= {len(ORIGINAL_BANK)} original + {len(PROMPT_CLAUSES)} prompt clauses + {len(NEGATIVE_CLAUSES)} negative clauses)")
print(f"SENTENCE_TEMPLATE: {SENTENCE_TEMPLATE!r}")
print(f"WORD_TEMPLATE    : {WORD_TEMPLATE!r}")
print()
print("--- ORIGINAL_BANK の cluster 構成 ---")
_prev = 0
for _name, _end in zip(CLUSTER_NAMES, CLUSTER_BOUNDARIES):
    print(f"  {_name:18s} : {_end - _prev} entries")
    _prev = _end
print()
print("--- wrap 後 (encode に渡される実際の文字列) サンプル ---")
for p in [ORIGINAL_BANK[0], ORIGINAL_BANK[7], PROMPT_CLAUSES[1], NEGATIVE_CLAUSES[0]]:
    print(f"  {SENTENCE_TEMPLATE.format(p=p)!r}")
print("--- prompt clauses (§3 の prompt を節分割) ---")
for c in PROMPT_CLAUSES:
    n_tok = len(pipe.tokenizer(c, add_special_tokens=False).input_ids)
    print(f"  [{n_tok:2d}t] {c!r}")
print("--- negative clauses (§3 の negative_prompt を節分割) ---")
for c in NEGATIVE_CLAUSES:
    n_tok = len(pipe.tokenizer(c, add_special_tokens=False).input_ids)
    print(f"  [{n_tok:2d}t] {c!r}")
print()

import numpy as np

pooled_rows = []
for p in PROMPT_BANK:
    p_wrapped = SENTENCE_TEMPLATE.format(p=p)
    with torch.no_grad():
        _, _, pooled_p, _ = pipe.encode_prompt(
            prompt=p_wrapped, device=torch.device(device), num_images_per_prompt=1,
            do_classifier_free_guidance=False,
        )
    pooled_rows.append(pooled_p[0].to(torch.float32).cpu().numpy())

pooled_arr = np.stack(pooled_rows, axis=0)
print(f"pooled_arr  : {pooled_arr.shape}   (N x 1280-d OpenCLIP-G pooled, sentences wrapped in SENTENCE_TEMPLATE)")


In [ ]:
import re
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

STOPWORDS = {"a", "an", "the", "of", "in", "on", "at", "to", "with",
             "and", "or", "for", "by"}
word_set: set[str] = set()
for p in PROMPT_BANK:
    for w in re.findall(r"[a-zA-Z]+", p.lower()):
        if w not in STOPWORDS:
            word_set.add(w)
word_list = sorted(word_set)
print(f"unique words after stopword removal: {len(word_list)}")
print(word_list)

# 単語側も同じ "a photo of ..." プレフィックスを共有 (WORD_TEMPLATE)。
# これで sentences / words が共通の caption-style に揃い、テンプレ方向のバイアスが
# 共通化されて cosine 距離から自然に抜ける (§5.2 末尾の解説参照)。
word_pooled_rows: list[np.ndarray] = []
for w in word_list:
    with torch.no_grad():
        _, _, pooled_p, _ = pipe.encode_prompt(
            prompt=WORD_TEMPLATE.format(w=w), device=torch.device(device), num_images_per_prompt=1,
            do_classifier_free_guidance=False,
        )
    word_pooled_rows.append(pooled_p[0].to(torch.float32).cpu().numpy())
word_pooled_arr = np.stack(word_pooled_rows, axis=0)
print(f"word_pooled_arr  : {word_pooled_arr.shape}   "
      f"(N words × 1280-d OpenCLIP-G pooled, template={WORD_TEMPLATE!r})")

combined_pooled = np.vstack([pooled_arr, word_pooled_arr])
n_sent = len(pooled_arr)
n_word = len(word_list)
n_total = n_sent + n_word

def pca_2d(X: np.ndarray) -> np.ndarray:
    Xc = X - X.mean(axis=0, keepdims=True)
    _, _, Vt = np.linalg.svd(Xc, full_matrices=False)
    return Xc @ Vt[:2].T

pca_combined = pca_2d(combined_pooled)
# t-SNE: perplexity を N/3 程度 + cosine 距離 で計算
# - embedding は norm 差が大きく Euclidean だと magnitude 支配で誇張されがち
# - 方向のみで測る cosine distance + perplexity ~ N/3 で「意味的に近い点」が近寄る
tsne_perplexity = max(5.0, min(30.0, n_total / 3.0))
tsne_combined = TSNE(
    n_components=2,
    perplexity=tsne_perplexity,
    metric="cosine",
    init="pca", random_state=seed, max_iter=1000,
).fit_transform(combined_pooled)
print(f"t-SNE: N={n_total} points, perplexity={tsne_perplexity:.1f}, metric=cosine")

def scatter_joint(ax, coords, title):
    ax.scatter(coords[:n_sent, 0], coords[:n_sent, 1],
               s=80, c="C0", alpha=0.85,
               edgecolors="white", linewidth=0.6,
               label=f"sentence (×{n_sent})", zorder=3)
    ax.scatter(coords[n_sent:, 0], coords[n_sent:, 1],
               s=35, c="C1", alpha=0.75, marker="^",
               label=f"word as \"a photo of a X\" (×{n_word})", zorder=2)
    for i in range(n_sent):
        lbl = PROMPT_BANK[i][:30] + ("…" if len(PROMPT_BANK[i]) > 30 else "")
        ax.annotate(lbl, (coords[i, 0], coords[i, 1]),
                    fontsize=6, xytext=(4, 3),
                    textcoords="offset points", color="C0", alpha=0.85)
    for j, w in enumerate(word_list):
        ax.annotate(w, (coords[n_sent + j, 0], coords[n_sent + j, 1]),
                    fontsize=6, xytext=(3, 2),
                    textcoords="offset points", color="C1", alpha=0.8)
    ax.set_title(title, fontsize=10)
    ax.grid(alpha=0.3)
    ax.legend(loc="best", fontsize=8)

fig, axes = plt.subplots(1, 2, figsize=(16, 7.5))
scatter_joint(axes[0], pca_combined,
              f"PCA (pooled, 1280-d) — {n_sent} sentences + {n_word} words")
scatter_joint(axes[1], tsne_combined,
              f"t-SNE (pooled, 1280-d, perplexity={tsne_perplexity:.0f}, cosine) "
              f"— {n_sent} sentences + {n_word} words")
fig.suptitle(
    "prompt embedding geometry — sentence + word joint map "
    f"(N={n_total} points in 1 space, unified \"a photo of ...\" template)",
    fontsize=11,
)
fig.tight_layout()
out_path = NB_OUT_DIR / "sec5_embedding_geometry.png"
fig.savefig(out_path, dpi=140)
print(f"saved: {out_path}")
plt.show()


### 5.4 pairwise cosine 距離 heatmap

PCA / t-SNE は **2D に潰す近似**なので、cluster 同士の相対距離は信頼しにくい (特に t-SNE)。**1280-d 空間での実際の距離**を直接見るために、22 文の **pairwise cosine 距離** を heatmap で表示します。

- 行・列の並びは **クラスタ順** (`ORIGINAL_BANK` の cluster A → E、その後に prompt clauses → negative clauses)
- 同じクラスタ内が暗い (= 近い) ブロック対角が見えれば、CLIP 空間で意図したクラスタが立っている証拠
- 白線が `ORIGINAL_BANK` の cluster 境界、橙線が clauses 境界、赤線が negative clauses 境界


In [ ]:
from sklearn.metrics import pairwise_distances

dist_mat = pairwise_distances(pooled_arr, metric="cosine")
sent_short = [p[:32] + ("…" if len(p) > 32 else "") for p in PROMPT_BANK]

fig, ax = plt.subplots(figsize=(9.5, 8.5))
im = ax.imshow(dist_mat, cmap="viridis_r", aspect="equal", vmin=0)
ax.set_xticks(range(len(PROMPT_BANK)))
ax.set_yticks(range(len(PROMPT_BANK)))
ax.set_xticklabels(sent_short, rotation=70, ha="right", fontsize=7)
ax.set_yticklabels(sent_short, fontsize=7)
ax.set_title(
    "pairwise cosine distance — pooled (1280-d), クラスタ順\n"
    "(対角ブロックが暗い = 同クラスタ内が近い)",
    fontsize=10,
)
cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cb.set_label("cosine distance (0=同方向, 1=直交)", fontsize=8)

n_orig = len(ORIGINAL_BANK)
n_clauses = len(PROMPT_CLAUSES)
# ORIGINAL_BANK 内の cluster 境界 (最後を除く)
for b in CLUSTER_BOUNDARIES[:-1]:
    ax.axhline(b - 0.5, color="white", linewidth=0.6)
    ax.axvline(b - 0.5, color="white", linewidth=0.6)
# ORIGINAL_BANK と prompt clauses の境界
ax.axhline(n_orig - 0.5, color="orange", linewidth=1.0)
ax.axvline(n_orig - 0.5, color="orange", linewidth=1.0)
# prompt clauses と negative clauses の境界
ax.axhline(n_orig + n_clauses - 0.5, color="red", linewidth=1.0)
ax.axvline(n_orig + n_clauses - 0.5, color="red", linewidth=1.0)

fig.tight_layout()
out_heat = NB_OUT_DIR / "sec5_cosine_distance_heatmap.png"
fig.savefig(out_heat, dpi=140)
print(f"saved: {out_heat}")
plt.show()

# クラスタ内/クラスタ間の平均距離を表示 (構造があるかの定量チェック)
intra = []
inter = []
cluster_id = []
_prev = 0
for _idx, _end in enumerate(CLUSTER_BOUNDARIES):
    cluster_id += [_idx] * (_end - _prev)
    _prev = _end
for i in range(n_orig):
    for j in range(i + 1, n_orig):
        if cluster_id[i] == cluster_id[j]:
            intra.append(dist_mat[i, j])
        else:
            inter.append(dist_mat[i, j])
print(f"ORIGINAL_BANK 内: 平均 cosine 距離  intra-cluster = {np.mean(intra):.3f}  inter-cluster = {np.mean(inter):.3f}")
print("-> intra < inter なら、5 クラスタ構造が pooled embedding 上で確かに立っている")


**観察ポイント**

- **pooled (1280-d, OpenCLIP-G の文全体表現)** は CLIP 系 contrastive 学習の結果として、似た意味の prompt 同士が近くに位置する傾向がある。本ノートでは **5 クラスタ (魔法使い / 動物 / 食べ物 / 乗り物 / 風景・画風) を意図的に並べた bank** にしてあるので、§5.4 heatmap の対角ブロックが暗くなれば「CLIP 空間に意図した cluster 構造が立っている」直接の証拠になる
- **テンプレート統一の効果**: 文も単語も `"a photo of ..."` プレフィックスで揃えて encode しているため、テンプレ方向のバイアスは共通項として効くだけになり、cosine 距離からはほぼキャンセルされる。これにより **「sentence 島 vs word 島」の 2 大島構造**が緩み、`witch` 単語が魔法使い系文の近傍に、`pizza` 単語が食べ物系文の近傍に来やすくなる (文と単語を同じ意味空間で比較可能になる)
- **t-SNE** は近傍関係を強調する非線形変換 — perplexity = N/3 + cosine 距離で「embedding の norm 差」に振り回されない設定。N=22 文 + 単語 ~70 で perplexity が ~30 まで上がる場合、cluster は緩く塊として見える程度になる (絶対距離は信頼しないこと)
- **fine 差ペア**: 同じ ORIGINAL_BANK 内の `witch ↔ wizard` (性別違いだけ) や、`watercolor painting of a forest ↔ realistic photo of a forest` (画風違いだけ) は、cluster A / E 内でも特に近い距離 (heatmap で対角の極所が最暗) になっているかを見ると、CLIP がどこまで微差を区別するかが分かる
- **prompt 節 (witch / forest / style)** は対応する ORIGINAL_BANK cluster の近くに着地するはず: 「witch flowing hair magic staff」節 → cluster A の近く、「enchanted forest with mushrooms」節 → cluster A & E (forest 系) の境界付近、「fantasy art masterpiece detailed」節 → cluster E (画風系) の近く
- **negative 節 (blurry / bad anatomy 等)** は ORIGINAL_BANK のどのクラスタにも属さない「低品質」概念で、scatter の中で **孤立 cluster** として浮き、heatmap では右下に暗いブロックを作って他の全 prompt と高い距離になりやすい

> **t-SNE の解釈に注意**: t-SNE は「近傍関係」を保ちやすいが、**点間の絶対距離・cluster の大きさ・cluster 同士の相対位置は信頼できない** (perplexity や random seed で大きく変わる)。「2 点が近い ↔ semantically 近い」と断言できるのは local neighborhood 内のみ、図全体は **探索的に眺める** 図として扱う。PCA は線形なので global 構造は読めるが、高次元空間の局所構造は落ちる。**cosine heatmap (§5.4) が一番ごまかしの少ない実距離の見方** で、PCA/t-SNE はその 2D 投影サマリ、という位置付け。


<a id="trajectory"></a>

---
## 6. 手動 scheduler ループで「ノイズから絵が立ち上がる」

ここからが 02 の主役です。`pipe(...)` 1 発呼びの代わりに **scheduler ループを手動で回し**、各 step での latent (4×128×128 のテンソル) を VAE で decode して、「絵がどう立ち上がるか」を時間軸で並べます。

### 全体構造 (もう一度)

```
prompt_embeds (1, 77, 2048) + pooled (1, 1280) + time/size 情報
                   │
                   ▼
latent_0 = N(0, σ²)  (1, 4, 128, 128)  ← ノイズ最大 (純ノイズ)
                   │ for i = 0 .. 29:
                   │     ε̂ = UNet(latent_i, t_i, text 条件)
                   │     (CFG なら 条件 + 無条件 を batch=2 で同時計算 →  ε̂ = ε̂_u + g·(ε̂_t − ε̂_u))
                   │     latent_{i+1} = scheduler.step(ε̂, t_i, latent_i)
                   ▼
latent_30 (clean)       (1, 4, 128, 128)
                   │
                   ▼ VAE.decode (latent → pixel)
image                   (3, 1024, 1024)
```

**step index の向き**: `i=0` がノイズ最大、`i=29` が最終 (DDPM 論文の `t=T → t=0` と逆順。3 年前 SD1.5 スライドもこの向き)。


### 6.1 CFG 用に prompt_embeds を batch 化


In [ ]:
# (CFG = Classifier-Free Guidance = 条件付き / 無条件の 2 本立てを差分で強調する手法。
#  unconditional は negative_prompt の embedding。本ノートでは guidance_scale=7.5 を採用
#  (SD1.5 系で歴史的に使われてきた典型値、SDXL Base の Diffusers 既定は 5.0)。
#  注意: Diffusers の `do_classifier_free_guidance = guidance_scale > 1` なので、
#  guidance_scale <= 1 のときは positive prompt のみで uncond path は計算されない。)

do_cfg = guidance_scale > 1.0
with torch.no_grad():
    pe, npe, pooled, npooled = pipe.encode_prompt(
        prompt=prompt,
        negative_prompt=negative_prompt or None,
        device=torch.device(device),
        num_images_per_prompt=1,
        do_classifier_free_guidance=do_cfg,
    )

# SDXL は addition_embed_type="text_time" なので「target_size / original_size /
# crop_top_left」も time embedding 経由で UNet に渡す (01 ノートブック §5 UNet 表参照)
add_time_ids = torch.tensor(
    [[height, width, 0, 0, height, width]],
    dtype=pe.dtype, device=device,
)
if do_cfg:
    # encode_prompt の戻り値で npe / npooled は do_cfg=False のとき None になる Optional 型。
    # ここは do_cfg=True の枝なので必ず Tensor だが、pyright が narrow できないので明示。
    assert npe is not None and npooled is not None
    prompt_embeds_in = torch.cat([npe, pe], dim=0)        # (2, 77, 2048)
    pooled_in        = torch.cat([npooled, pooled], dim=0) # (2, 1280)
    time_ids         = torch.cat([add_time_ids, add_time_ids], dim=0)  # (2, 6)
else:
    prompt_embeds_in, pooled_in, time_ids = pe, pooled, add_time_ids
added_cond_kwargs = {"text_embeds": pooled_in, "time_ids": time_ids}

print("guidance_scale       :", guidance_scale)
print("do_cfg               :", do_cfg)
print("prompt_embeds_in     :", tuple(prompt_embeds_in.shape))
print("added text_embeds    :", tuple(pooled_in.shape))
print("added time_ids       :", tuple(time_ids.shape))



### 6.2 scheduler の timestep 列と初期 latent


In [ ]:
pipe.scheduler.set_timesteps(num_inference_steps, device=device)
timesteps = pipe.scheduler.timesteps

print("scheduler          :", type(pipe.scheduler).__name__)
print("num_inference_steps:", num_inference_steps)
print("timesteps[:5]      :", [float(t) for t in timesteps[:5]])
print("timesteps[-5:]     :", [float(t) for t in timesteps[-5:]])
print("init_noise_sigma   :", float(pipe.scheduler.init_noise_sigma))


In [ ]:
# 初期 latent (純粋ガウシアンノイズ × init_noise_sigma)
unet = pipe.unet
latent_dtype = next(unet.parameters()).dtype
latent_shape = (1, unet.config.in_channels, height // 8, width // 8)

gen_device = "cpu" if device == "mps" else device  # winport fix(B-1): randn と generator の device を揃える (cuda 必須)
gen = torch.Generator(device=gen_device).manual_seed(seed)
latents = torch.randn(latent_shape, generator=gen, dtype=latent_dtype, device=gen_device).to(device)
latents = latents * pipe.scheduler.init_noise_sigma

print("latent shape    :", tuple(latents.shape))
print("latent dtype    :", latents.dtype)
print("latent mean/std :", f"{float(latents.mean()):+.3f} / {float(latents.std()):.3f}")


### 6.3 手動 loop — 中間 latent を VAE decode して trajectory grid に並べる


In [ ]:
import time
import math
from PIL import Image

def decode_latents_to_pil(latents: torch.Tensor) -> Image.Image:
    # latent (1, 4, H/8, W/8) -> PIL.Image (H, W, 3)
    # VAE.decode は scaling 処理しない、呼び出し側で /scaling_factor して渡す慣習
    vae = pipe.vae
    # winport fix(B-2): SDXL の fp16 VAE は decode で数値発散→黒画像。decode の間だけ fp32 へ局所 upcast し try/finally で戻す（恒久 upcast は pipe() を壊す）。
    _orig_vae_dtype = next(vae.parameters()).dtype
    _upcast = _orig_vae_dtype == torch.float16
    if _upcast:
        vae.to(torch.float32)
    try:
        with torch.no_grad():
            z = latents.to(dtype=next(vae.parameters()).dtype) / vae.config.scaling_factor
            img = vae.decode(z, return_dict=False)[0]   # (1, 3, H, W), range ~ [-1, 1]
    finally:
        if _upcast:
            vae.to(_orig_vae_dtype)
    img = (img / 2 + 0.5).clamp(0, 1)
    arr = (img[0].permute(1, 2, 0).float().cpu().numpy() * 255).astype(np.uint8)
    return Image.fromarray(arr)

# どの step を decode するか (i=29 が最終 clean latent)
# loop の前に「init」(scheduler.step 適用前の生 noise) を別途 capture するので
# decode_at から 0 は外す (step 0 = 1 step 適用後で init とは別物)
decode_at = [1, 3, 5, 10, 15, 20, 25, 29]
decoded: dict[int | str, Image.Image] = {}
latent_stats = []

# (NEW) loop 前に真の初期 noise (latent_0 = noise * init_noise_sigma) を decode
decoded["init"] = decode_latents_to_pil(latents)
init_sigma = float(pipe.scheduler.init_noise_sigma)
print(f"init noise: σ = {init_sigma:.2f}  (latent shape {tuple(latents.shape)})")

print(f"manual scheduler loop ({num_inference_steps} steps, decode at {decode_at})")
t0 = time.perf_counter()
for i, t in enumerate(timesteps):
    latent_model_input = torch.cat([latents] * 2) if do_cfg else latents
    latent_model_input = pipe.scheduler.scale_model_input(latent_model_input, t)
    with torch.no_grad():
        noise_pred = unet(
            latent_model_input, t,
            encoder_hidden_states=prompt_embeds_in,
            added_cond_kwargs=added_cond_kwargs,
            return_dict=False,
        )[0]
    if do_cfg:
        np_u, np_t = noise_pred.chunk(2)
        noise_pred = np_u + guidance_scale * (np_t - np_u)
    latents = pipe.scheduler.step(noise_pred, t, latents, return_dict=False)[0]

    lf = latents.detach().to(torch.float32).cpu()
    latent_stats.append({
        "step": i, "timestep": float(t),
        "mean": float(lf.mean()), "std": float(lf.std()),
        "min":  float(lf.min()),  "max": float(lf.max()),
    })

    if i in decode_at:
        decoded[i] = decode_latents_to_pil(latents)
        print(f"  step {i:3d}  t={float(t):7.1f}  "
              f"mean={float(lf.mean()):+.3f}  std={float(lf.std()):.3f}")

# 最終 latent は decoded[num_inference_steps - 1] = step 29 で既に capture 済
# (scheduler.step at i=29 後の clean latent、final と同一物なので別 key は作らない)
final_latents = latents.detach().clone()
elapsed_trajectory = time.perf_counter() - t0
print(f"trajectory done in {elapsed_trajectory:.1f} s")


In [ ]:
# trajectory grid を作る (1 行 × N 列、超横長で時間軸を明示)
# 新 ordering: init (scheduler.step 前) → step 1/3/.../29 (final)
numeric_steps = sorted([k for k in decoded if isinstance(k, int)])
ordered = ["init"] + numeric_steps
imgs = [decoded[k] for k in ordered]

def _label(k):
    if k == "init":
        return f"init (σ ≈ {init_sigma:.1f})"
    if k == numeric_steps[-1]:
        return f"step {k} (final)"
    return f"step {k}"

labels = [_label(k) for k in ordered]
n_panels = len(imgs)  # init + 8 decode points = 9

fig, axes = plt.subplots(1, n_panels, figsize=(n_panels * 2.0, 2.6))
axes = np.atleast_1d(axes)
for ax, img, lbl in zip(axes, imgs, labels):
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(lbl, fontsize=9)
fig.suptitle(
    f"denoising trajectory  (init → {num_inference_steps} scheduler steps、左端 = 生 noise、右端 = clean)",
    fontsize=11,
)
fig.tight_layout()
out_path = NB_OUT_DIR / "sec6_trajectory_grid.png"
fig.savefig(out_path, dpi=110)
print(f"saved: {out_path}  (1 × {n_panels} panel layout)")
plt.show()


In [ ]:
# 個別 PNG も保存しておく (decode_at + final)
for k, img in decoded.items():
    name = f"sec6_step_{k:03d}.png" if isinstance(k, int) else "sec6_final.png"
    img.save(NB_OUT_DIR / name)
print(f"saved {len(decoded)} individual PNGs to {NB_OUT_DIR}/")


In [ ]:
# 全 step の latent stats を CSV に書き出す (csv モジュールで明示的に)
import csv
csv_path = NB_OUT_DIR / "sec6_latent_stats.csv"
with csv_path.open("w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["step", "timestep", "mean", "std", "min", "max"])
    w.writeheader()
    for row in latent_stats:
        w.writerow(row)
print(f"saved: {csv_path}")


In [ ]:
# まとめプロット: step ごとの latent std / mean 推移
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
xs = [r["step"] for r in latent_stats]
axes[0].plot(xs, [r["std"] for r in latent_stats], "o-", ms=4)
axes[0].set_xlabel("step i (0 = most noisy)")
axes[0].set_ylabel("latent std")
axes[0].set_title("latent std (per step)")
axes[0].grid(alpha=0.3)
axes[1].plot(xs, [r["mean"] for r in latent_stats], "o-", ms=4, color="C1")
axes[1].set_xlabel("step i")
axes[1].set_ylabel("latent mean")
axes[1].set_title("latent mean (per step)")
axes[1].grid(alpha=0.3)
fig.tight_layout()
out_path = NB_OUT_DIR / "sec6_latent_stats.png"
fig.savefig(out_path, dpi=120)
print(f"saved: {out_path}")
plt.show()


### latent stats の定義 — mean / std は何を計算しているか

各 step で `latents` tensor の shape は **`(1, 4, 128, 128)`** = batch 1 × 4 channel × 128×128 spatial = **`N = 65,536` 個の scalar 要素**。`latent_stats` ではこの N 個を **全部混ぜた global mean / std** を取る (channel 別 / 空間別には分けない):

$$\mu_i = \frac{1}{N} \sum_{j=1}^{N} x_j^{(i)}, \qquad
\sigma_i = \sqrt{\frac{1}{N-1} \sum_{j=1}^{N} (x_j^{(i)} - \mu_i)^2}$$

ここで $i$ = step index (0..29)、$x_j^{(i)}$ = step $i$ の `latents` tensor を **flatten した j 番目の scalar 要素** (channel と空間位置を区別せず 1 つの要素として並べたもの)。

PyTorch コード対応: `latents.mean()` と `latents.std()` (引数なし) = 全要素の global scalar。なお `std` は不偏分散 (correction = 1、PyTorch default)。

> **note**: もし channel 別の挙動が知りたければ `latents.mean(dim=(0,2,3))` で 4 個の channel-wise mean が取れる。今回は **「latent 全体としての noise レベルがどう減るか」** を見たいので global で集計している。

---

**読み取れること**

#### trajectory 図 (上、1 row × 10 panel)

- **step 0-3**: 純ノイズ。decode しても色のかたまり程度しか見えない (= latent はまだ N(0, σ²) 由来の random noise が支配)
- **step 5-10**: 被写体の輪郭 (witch シルエット、staff の縦帯、forest 背景) が立ち上がり、配色レイアウトが固まる
- **step 15-25**: 細部の整形 (顔・髪・glow など)
- **step 29 = final**: 仕上がり画像 — VAE が出せる範囲での最終形

横一列に並べることで「**ノイズ → 構造 → 細部**」の遷移を時間軸として直接読める。

#### latent stats 図 (下、std と mean)

- **`latent std`**: `init_noise_sigma ≈ 11.5` (Euler scheduler 既定) から始まり、step が進むにつれ **単調減少** → 最終的に **≈ 0.87** に収束。このカーブには **2 つの regime** が混ざっている: (i) **前半 (sigma 大)** では `std ≈ scheduler.sigmas[i]` で scheduler の sigma schedule に近似一致、(ii) **後半 (sigma 小、~step 20 以降)** では scheduler の sigma は 0 へ向かうが、std は clean latent 自身の std (≈ 0.87、VAE の latent 統計由来) に **plateau**。つまり「scheduler だけ」の可視化ではなく、scheduler + UNet の noise prediction + CFG + prompt 条件 が複合した結果が見えている

- **`latent mean`**: 初期は **0 付近** (純 Gauss N(0, σ²) なので)、step 5-10 で急上昇して step 10 以降は **≈ 0.09 で安定**。mean が 0 から離れて安定すること自体は事実だが、**channel 別の意味づけは VAE の学習済み潜在表現次第** で、本ノートでは深追いしない (人間が読める輝度・色軸ではなく、4 channel の learned representation)

#### 2 枚を併せて読む

- 上の図 (trajectory) は **「人間が見て分かる」** 軸 (絵が立ち上がる過程)
- 下の図 (latent stats) は **「scheduler + UNet の数値挙動」** 軸 — 同じ過程を tensor の global statistics で見たもの
- std curve の急減期 (step 0-10) と trajectory の構造立ち上がり期が一致 ← **「数値の noise が消える期間 = 視覚的に絵が固まる期間」**

**`pipe(...)` を 1 発呼んだ場合と視覚的にほぼ同じ画像**が出ているはずです (同じ seed + scheduler + UNet processor)。[01ノートブック §3](01_sdxl_base_intro.ipynb#gen) の `sec3_base_30steps.png` と見比べるとほぼ一致します。ただし fp 精度・op 順序の違いで bit-exact 同一は保証外。


<a id="attn"></a>

---
## 7. cross-attention probe — 画像のどの位置がどの単語に注目しているか

UNet の中には **cross-attention** ブロックがたくさん入っていて、それぞれが「latent の各空間位置 (Query) × prompt の各 token (Key/Value)」の attention weight を計算しています。

**3 つの "向き" を意識する** (本節を読む前提):

- **attention 計算の向き — 画像 (Q) → 単語 (K/V)**: 画像のある位置 `q` が prompt のどの token `k` を見るかを softmax で測る (= attention 行列 `A[q, k]` の自然な読み方)
- **情報の流れ (V 経由) — 単語 → 画像**: V (text 側 hidden state) を attention 重みで重み付けして image 側に注入。画像生成における「テキストが画像を作る」機構そのもの
- **§7.5-7.8 で見るもの (= attention の逆解析) — 単語 → 画像**: 「単語 `witch` をよく見てる画像位置はどこか」を、attention 行列の **K 軸 (token) を witch の位置で固定** して Q 軸 (image) を 2D に並べることで可視化。**K 軸 (token) 固定 → Q 軸 (image) 2D 分布の読み方** (普通は q から k を見るのに、ここでは k 起点で q を解く)。結果として情報流の向き (単語→画像) と一致する

3 年前 (2023) の SD1.5 デモで見せていた「**画像のどの位置が `witch` という単語に注目しているか**」のヒートマップを SDXL Base で組み直します — まさに上の **逆解析 view**。§7.11 では **画像→単語 view** (Q 軸 image 位置を固定して K 軸 77 token 分布を見る) も並べて、両 view を比較できる構成にしてあります。

なお厳密には attention は **UNet 内部の latent grid 位置 (64×64 or 32×32)** 単位の重みで、画像上に重ねて表示する際は latent grid を画像解像度に upsample しています (§7.6 以降の panel title に解像度を表示)。

手順:

1. **§7.1**: UNet 内の cross-attention module を全列挙 (SDXL Base には 70 個ある) → 代表 11 module に絞る
2. **§7.2**: UNet 構造の解釈図 (.t0..tN の整列、head 計算数) — §7.6 以降の grid を読む前提
3. **§7.3**: token table — どの単語を heatmap の列として並べるか定義
4. **§7.4**: `RecordingAttnProcessor` を当てて 5 step (i=0, 7, 14, 22, 29) の attention weight を capture
5. **§7.5**: `(step, module, token) → 2D` 辞書 (heatmap_db) を構築
6. **§7.6**: 1 head (.h0) の raw attention を per-token grid で見る (witch / mushrooms)
7. **§7.7**: 各 layer の .t0 内の全 head を **PCA-RGB** に圧縮して per-token grid (witch / mushrooms)
8. **§7.8**: 全 t-block を pool して PCA-RGB (全 70 cross-attn modules × 全 head を 1 layer 内で集約)
9. **§7.9**: profile-based head finding — t0 module の attention 役割を 1 ベクトルで要約 (interesting t0 を最大 10 個選別)
10. **§7.10**: self-attention probe — §7.9 で選ばれた top-1 head 内で、各 word の cross-attn argmax 位置を query にした self-attn key 分布を可視化 (3 panel)
11. **§7.11**: cross-attention の 画像→単語 view — image 位置固定 → 77 token 分布 (EOS 以降の漏れも見る)


### 7.1 cross-attention module を列挙する


In [ ]:
import re
from diffusers.models.attention_processor import Attention

def parse_module_info(name: str) -> dict:
    # `down_blocks.1.attentions.0.transformer_blocks.0.attn2` のような
    # module 名から (stage, stage_index, attn_block_index, transformer_index) を抜く
    info = {"name": name, "stage": None, "stage_index": None,
            "attn_block_index": None, "transformer_index": None}
    m = re.search(r"(down_blocks|up_blocks|mid_block)\.?(\d+)?", name)
    if m:
        stage_raw, idx = m.group(1), m.group(2)
        info["stage"] = {"down_blocks": "down", "up_blocks": "up", "mid_block": "mid"}[stage_raw]
        if idx is not None:
            info["stage_index"] = int(idx)
    m = re.search(r"attentions\.(\d+)", name)
    if m:
        info["attn_block_index"] = int(m.group(1))
    m = re.search(r"transformer_blocks\.(\d+)", name)
    if m:
        info["transformer_index"] = int(m.group(1))
    return info

STAGE_ORDER = {"down": 0, "mid": 1, "up": 2}

def module_sort_key(info: dict) -> tuple:
    return (
        STAGE_ORDER.get(info["stage"], 99),
        info["stage_index"]      if info["stage_index"]      is not None else 99,
        info["attn_block_index"] if info["attn_block_index"] is not None else 99,
        info["transformer_index"] if info["transformer_index"] is not None else 99,
    )

def module_short_name(info: dict) -> str:
    parts = []
    if info["stage"] is not None:
        s = info["stage"]
        if info["stage_index"] is not None:
            s = f"{s}{info['stage_index']}"
        parts.append(s)
    if info["attn_block_index"] is not None:
        parts.append(f"a{info['attn_block_index']}")
    if info["transformer_index"] is not None:
        parts.append(f"t{info['transformer_index']}")
    return ".".join(parts) if parts else info["name"]

all_cross = []
for name, mod in pipe.unet.named_modules():
    if isinstance(mod, Attention) and getattr(mod, "is_cross_attention", False):
        info = parse_module_info(name)
        info["heads"] = int(getattr(mod, "heads", 0) or 0)
        all_cross.append(info)

n_by_stage = {"down": 0, "mid": 0, "up": 0}
for r in all_cross:
    n_by_stage[r["stage"]] = n_by_stage.get(r["stage"], 0) + 1
print(f"全 cross-attention modules: {len(all_cross)}")
print(f"  down: {n_by_stage['down']}    mid: {n_by_stage['mid']}    up: {n_by_stage['up']}")


In [ ]:
# 各 (stage, stage_index, attn_block_index) グループから transformer_index=0 を代表として選ぶ
groups: dict[tuple, list[dict]] = {}
for r in all_cross:
    k = (r["stage"], r["stage_index"], r["attn_block_index"])
    groups.setdefault(k, []).append(r)

selected = []
for k, group in groups.items():
    group.sort(key=lambda r: r["transformer_index"] or 0)
    selected.append(group[0])
selected.sort(key=module_sort_key)

print(f"selected 代表 modules: {len(selected)}  (各グループから transformer_index=0 を 1 つずつ)")
print()
print(f"  {'#':>3s}  {'short':12s}  {'stage':<5s}  {'sIdx':>5s}  {'aIdx':>5s}  {'heads':>5s}")
print(f"  {'-'*3}  {'-'*12}  {'-'*5}  {'-'*5}  {'-'*5}  {'-'*5}")
for i, r in enumerate(selected):
    print(f"  {i:3d}  {module_short_name(r):12s}  {r['stage']:<5s}  "
          f"{str(r['stage_index']):>5s}  {str(r['attn_block_index']):>5s}  {r['heads']:>5d}")


**観察 ([01ノートブック §5](01_sdxl_base_intro.ipynb#inside) UNet 表とつなげて)**: SDXL Base の cross-attention は **down 2 段 + mid 1 段 + up 2 段** に分布しています ([01ノートブック §5](01_sdxl_base_intro.ipynb#inside) の `down_block_types`:`[Down, CrossAttnDown, CrossAttnDown]`、`up_block_types`:`[CrossAttnUp, CrossAttnUp, Up]` のとおり、stage 0 の最浅は cross-attention を持たない)。

更に SDXL の特徴として **同一 `(stage, stage_index, attn_block_index)` の中に複数の transformer block が並ぶ** ことがあります (down2 / mid / up0 にはそれぞれ ~10 個ずつ)。ここでは可視化のために各グループの先頭 (`transformer_index=0`) を代表として 11 個に絞っています。

cross-attention の空間解像度は **query 列長 (latent の H×W)** から決まり、SDXL Base 1024 生成時は **{64×64, 32×32} の 2 種類だけ**になります (SD1.5 は 64/32/16/8 の 4 種、3 年前スライドの「6 列展開」は SDXL では 4 列〜5 列に縮む)。


---
### 7.2 U-Net 内 cross-attention の処理順 — 全体像

§7.6 / §7.7 で **11 module を縦に並べる** ことになるが、その「11 module」とは何かを最初に押さえておく。実際の U-Net 内では、cross-attention は以下の順序で何回も呼ばれる:

```
input latents
  ↓ down_blocks.0 (cross-attn なし、スキップ)
  ↓ down_blocks.1 → down1.a0.[t0,t1]      ←  cross-attn 2 layer × 2 t-block = 4 計算
  ↓                 down1.a1.[t0,t1]
  ↓ down_blocks.2 → down2.a0.[t0..t9]     ←  cross-attn 2 layer × 10 t-block = 20 計算
  ↓                 down2.a1.[t0..t9]
  ↓ mid_block     → mid.a0.[t0..t9]       ←  cross-attn 1 layer × 10 t-block = 10 計算
  ↓ up_blocks.0   → up0.a0.[t0..t9]       ←  cross-attn 3 layer × 10 t-block = 30 計算
                    up0.a1.[t0..t9]
                    up0.a2.[t0..t9]
  ↓ up_blocks.1   → up1.a0.[t0,t1]        ←  cross-attn 3 layer × 2 t-block = 6 計算
                    up1.a1.[t0,t1]
                    up1.a2.[t0,t1]
  ↓ up_blocks.2 (cross-attn なし)
output (clean noise prediction)
```

各 `[t0..tN]` の中身は **直列に stack された transformer block 群** で、各 t-block 内で `multi-head attention (heads = 10 or 20)` が走る (`.t0` の出力 → `.t1` の入力 → ... → `.tN` の入力)。

#### attention layer 別の規模

| attention layer | t-blocks 数 | heads/block | layer 全 cross-attn module 数 | head 計算数 (1 forward pass) |
|---|---:|---:|---:|---:|
| down1.a0, .a1 | 2 each | 10 | 4 | 40 |
| down2.a0, .a1 | **10 each** | 20 | 20 | 400 |
| mid.a0 | **10** | 20 | 10 | 200 |
| up0.a0, .a1, .a2 | **10 each** | 20 | 30 | 600 |
| up1.a0, .a1, .a2 | 2 each | 10 | 6 | 60 |
| **合計** | — | — | **70 modules** | **~1300 個 / forward pass** |

例えば `down2.a0` だけで:
- 10 個の transformer block が **直列** (`.t0` → `.t9`、出力 → 入力)
- 各 block の cross-attn は **20 head 並列**
- 合計 **10 × 20 = 200 個の独立 cross-attn head 計算** が 1 回の forward pass で実行

U-Net 全体では cross-attn だけで **約 1300 個** (40 + 400 + 200 + 600 + 60) の head 計算が **1 step あたり**走り、それを 30 step 繰り返す。

#### §7.6 / §7.7 が「11 module」とする縮約

「全 70 modules ✕ 全 head」を可視化すると figure が膨れ過ぎる (700 panel × token 数)。§7.6 / §7.7 では **各 `(stage, stage_index, attn_block_index)` グループから `.t0` (最初の transformer block) を 1 つ代表として選んだ 11 modules** に絞り、さらに head 平均 (or PCA 圧縮) で表示する:

- §7.6: 11 modules × `.t0` の **head 0 だけ** を per-token grid に
- §7.7: 11 modules × `.t0` の **全 head を PCA-RGB 圧縮** して per-token grid に
- §7.8: 「もし `.t0..tN` 全部 pool したら?」(= 200 heads/layer 規模) を本ノートで in-line に実装

つまり「11 module 縦並び」は **全 70 modules のうち代表 11 個** を可視化に使っている、と覚えておけば §7.6 以降の grid 解釈に迷わない。


### 7.3 token table — 列として並べる単語の定義


In [ ]:
# 魔法使い prompt の主要 token を target にする
TARGET_TOKENS = ["witch", "hair", "staff", "forest", "mushrooms"]
TARGET_PHRASES = [
    ("magic_staff",              ["magic", "staff"]),
    ("enchanted_forest",         ["enchanted", "forest"]),
    ("bioluminescent_mushrooms", ["bioluminescent", "mushrooms"]),
]

def build_token_table(tokenizer, prompt: str,
                      target_words: list[str],
                      phrases: list[tuple[str, list[str]]]) -> dict:
    ids = tokenizer(prompt, padding="max_length",
                    max_length=tokenizer.model_max_length,
                    truncation=True, return_tensors="pt").input_ids[0].tolist()
    tokens = tokenizer.convert_ids_to_tokens(ids)

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # 【核心】word → token-index-set 変換 (§7 in-context probe の本質)
    #
    # この関数が返す list[int] は「target word が 77-token の実プロンプト列の
    # どの位置に居るか」の subtoken 単位 添字集合 (具体値は本セル末尾の print
    # 出力で TARGET_TOKENS / TARGET_PHRASES の実 indices として確認できる)。
    #
    # 後段 §7.5 では cross-attention 行列 (Q=画像位置, K=77 token 位置) の
    # K 軸の中から **この添字集合の列だけ** を取り出し、image 側 2D に reshape
    # して per-token heatmap にする。つまり §7 で見ているのは:
    #
    #   ✗  文脈から切り離した "mushrooms" 単独 embedding に対する attention
    #   ○  実プロンプト「a young witch ... bioluminescent mushrooms ...」内の
    #      "mushrooms" が居る位置 (BPE 5 subtoken 連続) に対する attention
    #
    # ── 「単語そのもの」ではなく **「プロンプト内の文脈付き位置」** への注目度
    # を可視化していることに注意。帰結:
    #   - その位置の hidden state は CLIP の causal Transformer 経由で BOS と
    #     先行 token の context を吸い込んでおり、辞書的な単語意味とは別物
    #   - 同じ語でも別 prompt なら別の attention map になる
    #   - 多 subtoken 語 (bioluminescent = bi+olu+min+esc+ent の 5 subtoken)
    #     は span 全体の添字を返し、§7.5 で K 列を **合算** する
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    def find_subtoken_indices(word: str) -> list[int]:
        # word を BPE で encode した subtoken 列を、prompt 全体の token 列の中で
        # contiguous span として一致探索。with-space / no-space の両 candidate で
        # 探し、見つかった span 全体 (= 該当する全 subtoken 位置) を返す。
        # 旧実装 (`cand[0]` の単独 token search) では multi-subtoken 語
        # (例: bioluminescent = bi + olu + min + esc + ent の 5 subtoken) で
        # 先頭 subtoken の位置 1 個しか返らず、後段の attention sum が
        # phrase の subtoken の一部しかカバーしない bug があった。
        hits = []
        candidates = [
            tokenizer(" " + word, add_special_tokens=False).input_ids,
            tokenizer(word, add_special_tokens=False).input_ids,
        ]
        for cand in candidates:
            if not cand:
                continue
            L = len(cand)
            for i in range(len(ids) - L + 1):
                if ids[i:i + L] == cand:
                    hits.extend(range(i, i + L))
        return sorted(set(hits))

    token_index = {w: find_subtoken_indices(w) for w in target_words}
    phrase_index = {}
    for name, parts in phrases:
        idxs = []
        for p in parts:
            idxs += find_subtoken_indices(p)
        phrase_index[name] = sorted(set(idxs))
    return {"ids": ids, "tokens": tokens,
            "token_index": token_index, "phrase_index": phrase_index}

ttab = build_token_table(pipe.tokenizer, prompt, TARGET_TOKENS, TARGET_PHRASES)

print(f"prompt: {prompt}")
print()
print(f"tokens[:25] : {ttab['tokens'][:25]}")
print()
print("target tokens (CLIP-L tokenizer での全 subtoken 位置):")
for w in TARGET_TOKENS:
    idxs = ttab["token_index"].get(w, [])
    mark = "✓" if idxs else "✗ not found"
    print(f"  {w:25s} → {idxs}  {mark}")
print()
print("target phrases (構成 word の全 subtoken の attention 列を加算):")
for pname, idxs in ttab["phrase_index"].items():
    mark = "✓" if idxs else "✗ empty"
    print(f"  {pname:25s} → {idxs}  {mark}")


### 7.4 `RecordingAttnProcessor` を当てて 5 step の attention を capture

`diffusers` の `AttnProcessor` を写経して、cross-attention の `attention_probs` (= softmax 後の attention weight) を「ON/OFF flag」で選択的に capture する processor を作ります。

> **note (observation forward)**: §7.4 以降の forward pass は **attention processor を観察用に差し替えて** 走らせます:
>
> - capture 対象 module → `RecordingAttnProcessor` (attention weight を保存する plain 実装)
> - 他の module → `AttnProcessor()` (plain、SDPA を使わない default 実装)
>
> 数学的には元の `AttnProcessor2_0` (SDPA ベース、Diffusers の load 時 default) と等価ですが、SDPA の op 融合や fp 累積精度の違いで bit-exact 同一には **ならない**。**観察用 forward** として、§6 の生成結果と数値レベルで完全一致するわけではない点に注意。


In [ ]:
from diffusers.models.attention_processor import AttnProcessor

class RecordingAttnProcessor(AttnProcessor):
    # cross-attention の attention_probs を capture_now_flag が ON のときだけ保存。
    # classic AttnProcessor の forward を写経 (math 同一)。
    # AttnProcessor を継承する: pyright で attn.processor への代入時に nominal 型一致が要求されるため。

    def __init__(self, name: str, info: dict, capture_now_flag: dict, captured: list):
        super().__init__()   # AttnProcessor は __init__ なしなので no-op
        self.name = name
        self.info = info
        self.capture_now_flag = capture_now_flag
        self.captured = captured

    def __call__(self, attn, hidden_states, encoder_hidden_states=None,
                 attention_mask=None, temb=None, *args, **kwargs):
        # SD1.5 / SDXL の通常 self/cross-attention では to_k / to_v / to_out は必ず存在する。
        # Pyright 用に narrowing しておく (Diffusers の Attention は joint cross attention
        # 等を見越して to_k / to_v / to_out を Optional 型で保持している)。
        assert attn.to_k is not None and attn.to_v is not None and attn.to_out is not None
        residual = hidden_states
        input_ndim = hidden_states.ndim
        if input_ndim == 4:
            batch_size, channel, h, w = hidden_states.shape
            hidden_states = hidden_states.view(batch_size, channel, h * w).transpose(1, 2)
        batch_size, seq_len, _ = (
            hidden_states.shape if encoder_hidden_states is None else encoder_hidden_states.shape
        )
        attention_mask = attn.prepare_attention_mask(attention_mask, seq_len, batch_size)  # pyright: ignore[reportArgumentType]
        if attn.group_norm is not None:
            hidden_states = attn.group_norm(hidden_states.transpose(1, 2)).transpose(1, 2)
        query = attn.to_q(hidden_states)
        is_cross = encoder_hidden_states is not None
        if encoder_hidden_states is None:
            encoder_hidden_states = hidden_states
        elif attn.norm_cross:
            encoder_hidden_states = attn.norm_encoder_hidden_states(encoder_hidden_states)
        key = attn.to_k(encoder_hidden_states)
        value = attn.to_v(encoder_hidden_states)
        query = attn.head_to_batch_dim(query)
        key = attn.head_to_batch_dim(key)
        value = attn.head_to_batch_dim(value)
        attention_probs = attn.get_attention_scores(query, key, attention_mask)

        if is_cross and self.capture_now_flag.get("on", False):
            self.captured.append({
                "name": self.name,
                "info": self.info,
                "shape": list(attention_probs.shape),
                "probs": attention_probs.detach().to(torch.float32).cpu().clone(),
                "heads": int(attn.heads),
                "batch": int(batch_size),
                "query_len": int(attention_probs.shape[1]),
                "key_len":   int(attention_probs.shape[2]),
            })

        hidden_states = torch.bmm(attention_probs, value)
        hidden_states = attn.batch_to_head_dim(hidden_states)
        hidden_states = attn.to_out[0](hidden_states)
        hidden_states = attn.to_out[1](hidden_states)
        if input_ndim == 4:
            hidden_states = hidden_states.transpose(-1, -2).reshape(batch_size, channel, h, w)
        if attn.residual_connection:
            hidden_states = hidden_states + residual
        hidden_states = hidden_states / attn.rescale_output_factor
        return hidden_states


In [ ]:
# selected module だけ RecordingAttnProcessor に差し替え、その他は素の AttnProcessor に
capture_now_flag = {"on": False}
captured_buffer: list[dict] = []
original_procs: dict = {}
selected_names = {r["name"] for r in selected}

for name, mod in pipe.unet.named_modules():
    if isinstance(mod, Attention):
        original_procs[name] = mod.processor
        if name in selected_names:
            mod.processor = RecordingAttnProcessor(
                name, parse_module_info(name), capture_now_flag, captured_buffer)
        else:
            mod.processor = AttnProcessor()

print(f"processor swap: {len(original_procs)} attention modules total, "
      f"{len(selected_names)} を Recording に差し替え")


In [ ]:
# どの step で capture するか — i=0 / 25% / 50% / 75% / 最終
capture_steps = [0, 7, 14, 22, 29]
print(f"capture_steps : {capture_steps}  (i=0 ノイズ最大, i=29 最終)")

# 同じ初期 latent + 同じ scheduler 設定で manual loop を再走 (今度は attention 付き)
pipe.scheduler.set_timesteps(num_inference_steps, device=device)
timesteps = pipe.scheduler.timesteps
gen_device = "cpu" if device == "mps" else device  # winport fix(B-1): randn と generator の device を揃える (cuda 必須)
gen2 = torch.Generator(device=gen_device).manual_seed(seed)
latents = torch.randn(latent_shape, generator=gen2, dtype=latent_dtype, device=gen_device).to(device)
latents = latents * pipe.scheduler.init_noise_sigma

step_captured: dict[int, list[dict]] = {}
decoded_at_capture: dict[int, Image.Image] = {}

print(f"manual loop with attention capture ({num_inference_steps} steps)")
t0 = time.perf_counter()
for i, t in enumerate(timesteps):
    latent_model_input = torch.cat([latents] * 2) if do_cfg else latents
    latent_model_input = pipe.scheduler.scale_model_input(latent_model_input, t)
    capture_now_flag["on"] = i in capture_steps
    captured_buffer.clear()
    with torch.no_grad():
        noise_pred = unet(
            latent_model_input, t,
            encoder_hidden_states=prompt_embeds_in,
            added_cond_kwargs=added_cond_kwargs,
            return_dict=False,
        )[0]
    if do_cfg:
        np_u, np_t = noise_pred.chunk(2)
        noise_pred = np_u + guidance_scale * (np_t - np_u)
    latents = pipe.scheduler.step(noise_pred, t, latents, return_dict=False)[0]

    if i in capture_steps and captured_buffer:
        step_captured[i] = [dict(c) for c in captured_buffer]
        decoded_at_capture[i] = decode_latents_to_pil(latents)
        print(f"  step {i:3d} captured {len(captured_buffer):3d} attention maps  "
              f"(query_len list: {sorted({c['query_len'] for c in captured_buffer})})")

# 元の processor に戻す
for name, mod in pipe.unet.named_modules():
    if isinstance(mod, Attention) and name in original_procs:
        mod.processor = original_procs[name]

print(f"capture done in {time.perf_counter() - t0:.1f} s, "
      f"restored {len(original_procs)} original processors")


### 7.5 heatmap データを構築 — `(step, module, token) → 2D` の辞書を作る


In [ ]:
# attention_probs は (B*heads, Q, K)。head 平均 + CFG 時は text 側 (batch index 1) のみを使う。
heatmap_db: dict[tuple, np.ndarray] = {}
module_res_db: dict[str, tuple[int, int]] = {}

for step_idx, recs in step_captured.items():
    for rec in recs:
        name = rec["name"]
        probs = rec["probs"]  # (B*heads, Q, K)
        heads = rec["heads"]
        bhi = probs.shape[0]
        n_batch = bhi // heads
        p = probs.view(n_batch, heads, probs.shape[1], probs.shape[2])
        if n_batch == 2:
            p = p[1:2]            # CFG: text 側 (positive) のみ
        p_avg = p.mean(dim=1)[0]  # (Q, K)、heads 平均

        Q = p_avg.shape[0]
        side = int(round(math.sqrt(Q)))
        if side * side != Q:
            continue
        module_res_db[name] = (side, side)

        # per-token: 単語の token index 列の attention 列を加算
        for word, idxs in ttab["token_index"].items():
            if not idxs:
                continue
            cols = p_avg[:, idxs].sum(dim=1)
            heatmap_db[(step_idx, name, word)] = cols.view(side, side).numpy()
        # per-phrase
        for pname, idxs in ttab["phrase_index"].items():
            if not idxs:
                continue
            cols = p_avg[:, idxs].sum(dim=1)
            heatmap_db[(step_idx, name, pname)] = cols.view(side, side).numpy()

print(f"heatmap_db: {len(heatmap_db)} entries  (= step × module × token/phrase の積)")
print()
print("取得できた spatial resolution:")
res_seen: dict[tuple[int, int], list[str]] = {}
for name, res in module_res_db.items():
    res_seen.setdefault(res, []).append(name)
for res in sorted(res_seen.keys()):
    print(f"  {res[0]:3d} × {res[1]:3d}  ({len(res_seen[res])} module)")

# §7.6 / §7.7 / §7.8 で参照する「最終 capture step」を確定
last_step = max(step_captured.keys())
print(f"last_step = {last_step}  (最終 capture step、§7.6 以降で使用)")


In [ ]:
# row_specs: §7.7 系で各 token を行として並べるための (label, key) リスト
row_specs: list[tuple[str, str]] = []
for w in TARGET_TOKENS:
    if ttab["token_index"].get(w):
        row_specs.append((w, w))
for pname, idxs in ttab["phrase_index"].items():
    if idxs:
        row_specs.append((pname.replace("_", " "), pname))
print(f"row_specs ({len(row_specs)} entries):")
for label, key in row_specs:
    print(f"  {label!r} → key={key!r}")


---
### 7.6 1 inner head の raw attention を per-token grid で見る (§7.8 への入門)

§7.7 では各 layer の **head 平均** を 1 枚に集約していた (10 or 20 head を 1 つに圧縮した代表 map)。ここでは **頭平均をやめ、各 layer の inner head **`head 0`** だけ** を 取り出して、token 単位の per-token grid を作る。

これは §7.8 で扱う「全 t-block × 全 head の PCA-RGB 集約」の **最も単純な構成要素** — head 1 個分の生 attention map。head 平均でも PCA でもなく、「1 head が出す attention 分布の素の姿」を素直に見るための入門位置付け。

**逆解析 view の典型例**: ここで作る heatmap はまさに「**witch をよく見てる画像位置はどこか**」 — attention 行列 `A[q, k=witch_subtoken_indices]` を画像 2D に reshape した、§7 冒頭で整理した **attention の逆解析** の最も素朴な姿 (head 1 個 / step 5 個 / 11 layer の grid)。

- **token**: witch / mushrooms の 2 個に絞る (compact + 主役 + 視覚的明瞭性)
- **head**: head 0 固定 (1 layer 1 head)
- **layout (§7.8 と同じ)**: 1 PNG / token、rows = 5 capture steps、cols = 11 layers (.t0 のみ) + 1 decoded
- 出力: `sec7_6_per_token_head0_<key>.png` × 2


In [ ]:
from matplotlib import cm

SEC77B_TARGET_TOKENS = [("witch", "witch"), ("mushrooms", "mushrooms")]
SEC77B_HEAD_IDX = 0

def get_head_token_map(step_idx: int, module_name: str, key: str, head_idx: int = SEC77B_HEAD_IDX):
    # 戻り値: (side, side) numpy heatmap または None
    rec = next((r for r in step_captured[step_idx] if r["name"] == module_name), None)
    if rec is None:
        return None
    probs = rec["probs"]   # (B*heads, Q, K)
    heads_n = rec["heads"]
    if head_idx >= heads_n:
        return None
    nb_ = probs.shape[0] // heads_n
    p = probs.view(nb_, heads_n, probs.shape[1], probs.shape[2])
    if nb_ == 2:
        p = p[1:2]  # CFG text side
    p_head = p[0, head_idx]  # (Q, K)
    idxs = ttab["token_index"].get(key) or ttab["phrase_index"].get(key) or []
    if not idxs:
        return None
    cols = p_head[:, idxs].sum(dim=1)  # (Q,)
    side = int(round(p_head.shape[0] ** 0.5))
    if side * side != p_head.shape[0]:
        return None
    return cols.view(side, side).numpy()


def render_head_per_token(label: str, key: str, head_idx: int = SEC77B_HEAD_IDX) -> Path:
    steps_avail = sorted(step_captured.keys())
    n_modules = len(selected)
    n_cols = n_modules + 1   # +1 for decoded image column
    n_steps = len(steps_avail)
    fig, axes = plt.subplots(n_steps, n_cols, figsize=(n_cols * 1.6, n_steps * 1.6))
    axes = np.atleast_2d(axes)

    for ri, step_idx in enumerate(steps_avail):
        for cj, mr in enumerate(selected):
            ax = axes[ri, cj]
            ax.axis("off")
            arr = get_head_token_map(step_idx, mr["name"], key, head_idx)
            if arr is None:
                ax.set_facecolor("#222")
                continue
            a = arr.copy()
            if a.max() > a.min():
                a = (a - a.min()) / (a.max() - a.min())
            ax.imshow(a, cmap="inferno")
            if ri == 0:
                res = module_res_db.get(mr["name"], (0, 0))
                short = module_short_name(mr)  # keep .t0 suffix
                heads_total = mr.get("heads", "?")
                ax.set_title(f"{short}\n({res[0]}×{res[1]})\nheads={heads_total}",
                             fontsize=7)
            if cj == 0:
                ax.text(-0.20, 0.5, f"step {step_idx}",
                        transform=ax.transAxes, fontsize=8,
                        ha="right", va="center")
        # decoded image at this step
        ax_im = axes[ri, -1]
        ax_im.axis("off")
        img_dec = decoded_at_capture.get(step_idx)
        if img_dec is not None:
            ax_im.imshow(img_dec)
        if ri == 0:
            ax_im.set_title("decoded\nat step", fontsize=7)

    fig.suptitle(
        f"token = '{label}' — head {head_idx} of each layer (t0) × {n_steps} capture steps  "
        f"(raw attention, no head averaging)",
        fontsize=9, y=0.993,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.985))
    fname_key = key.replace(" ", "_")
    out = NB_OUT_DIR / f"sec7_6_per_token_head{head_idx}_{fname_key}.png"
    fig.savefig(out, dpi=130)
    plt.show()
    return out


print(f"rendering §7.6 head{SEC77B_HEAD_IDX} per-token grids: {len(SEC77B_TARGET_TOKENS)} tokens")
for label, key in SEC77B_TARGET_TOKENS:
    print(f"  '{key}' ...")
    render_head_per_token(label, key)


---
### 7.7 head PCA-RGB (.t0 only) を per-token grid で見る (§7.8 への入門 part 2)

§7.6 では **1 head の raw attention** を見た。ここでは **同 layer の `.t0` 内の 10 or 20 head 全部** を head 軸で並べて **PCA で 3 次元に圧縮** → R/G/B に割り当てる。これは §7.8 の **`.t0` 限定の小型版** に相当。

§7.8 との対比:

| | §7.7 (本セクション) | §7.8 |
|---|---|---|
| 取り出す範囲 | 各 layer の **.t0 のみ** (10 or 20 heads) | 各 layer の **全 .t0..tN** (down2 なら 200 heads) |
| 観察できること | layer 内 head の協調パターン | layer 全体 (深さ方向) の head 構造 |

- **token**: witch / mushrooms の 2 個
- **layout (§7.8 と同じ)**: 1 PNG / token、rows = 5 capture steps、cols = 11 layers (.t0) + 1 decoded
- **PCA 処理**: §7.8 と同じ (sign 補正 + sqrt(ref) brightness modulation)
- 出力: `sec7_7_per_token_<key>.png` × 2


In [ ]:
from matplotlib import cm
from PIL import Image

SEC77C_TARGET_TOKENS = [("witch", "witch"), ("mushrooms", "mushrooms")]

def pca_rgb_for_module_token(module_name: str, key: str, step_idx: int):
    # .t0 のみの PCA-RGB (sign 補正 + sqrt(ref) brightness)
    # 戻り値: (rgb (H, W, 3) in [0,1], top3 var ratio) または (None, None)
    rec = next((r for r in step_captured[step_idx] if r["name"] == module_name), None)
    if rec is None:
        return None, None
    probs = rec["probs"]
    heads_n = rec["heads"]
    nb_ = probs.shape[0] // heads_n
    p = probs.view(nb_, heads_n, probs.shape[1], probs.shape[2])
    if nb_ == 2:
        p = p[1:2]
    idxs = ttab["token_index"].get(key) or ttab["phrase_index"].get(key) or []
    if not idxs:
        return None, None
    p_token = p[0, :, :, idxs].sum(dim=2)  # (heads, Q)
    side = int(round(p_token.shape[1] ** 0.5))
    if side * side != p_token.shape[1]:
        return None, None
    ref = p_token.mean(dim=0).numpy().astype(np.float32)
    X = p_token.numpy().T.astype(np.float32)
    Xc = X - X.mean(axis=0, keepdims=True)
    try:
        _, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    except np.linalg.LinAlgError:
        return None, None
    n_pc = min(3, Vt.shape[0])
    Y = Xc @ Vt[:n_pc].T
    ref_c = ref - ref.mean()
    for k in range(n_pc):
        if float(np.dot(Y[:, k], ref_c)) < 0:
            Y[:, k] = -Y[:, k]
    if n_pc < 3:
        Y_full = np.zeros((Y.shape[0], 3), dtype=np.float32)
        Y_full[:, :n_pc] = Y
        Y = Y_full
    Y_min = Y.min(axis=0)
    Y_max = Y.max(axis=0)
    Y_norm = (Y - Y_min) / (Y_max - Y_min + 1e-9)
    ref_norm = (ref - ref.min()) / (ref.max() - ref.min() + 1e-9)
    rgb = Y_norm.reshape(side, side, 3) * np.sqrt(ref_norm).reshape(side, side, 1)
    var_total = float((S ** 2).sum())
    var_top3 = float((S[:n_pc] ** 2).sum())
    return rgb, var_top3 / max(var_total, 1e-9)


def render_pca_rgb_per_token(label: str, key: str) -> Path:
    steps_avail = sorted(step_captured.keys())
    n_modules = len(selected)
    n_cols = n_modules + 1   # +1 for decoded
    n_steps = len(steps_avail)
    fig, axes = plt.subplots(n_steps, n_cols, figsize=(n_cols * 1.6, n_steps * 1.6))
    axes = np.atleast_2d(axes)

    for ri, step_idx in enumerate(steps_avail):
        for cj, mr in enumerate(selected):
            ax = axes[ri, cj]
            ax.axis("off")
            rgb, var3 = pca_rgb_for_module_token(mr["name"], key, step_idx)
            if rgb is None:
                ax.set_facecolor("#222")
                continue
            assert var3 is not None
            rgb_u8 = (rgb * 255).astype(np.uint8)
            pil = Image.fromarray(rgb_u8).resize((128, 128), Image.Resampling.NEAREST)
            ax.imshow(pil)
            if ri == 0:
                res = module_res_db.get(mr["name"], (0, 0))
                short = module_short_name(mr)  # keep .t0 suffix
                heads_n = mr.get("heads", "?")
                ax.set_title(f"{short}\n({res[0]}×{res[1]})\nheads={heads_n}",
                             fontsize=7)
            if cj == 0:
                ax.text(-0.20, 0.5, f"step {step_idx}",
                        transform=ax.transAxes, fontsize=8,
                        ha="right", va="center")
            ax.text(0.97, 0.02, f"{var3*100:.1f}%",
                    transform=ax.transAxes, fontsize=6,
                    ha="right", va="bottom", color="white",
                    bbox=dict(facecolor="black", alpha=0.55, pad=0.8,
                              edgecolor="none"))
        # decoded image at this step
        ax_im = axes[ri, -1]
        ax_im.axis("off")
        img_dec = decoded_at_capture.get(step_idx)
        if img_dec is not None:
            ax_im.imshow(img_dec)
        if ri == 0:
            ax_im.set_title("decoded\nat step", fontsize=7)

    fig.suptitle(
        f"token = '{label}' — layer t0 only, PCA-RGB (10 or 20 heads per layer) × {n_steps} capture steps  "
        f"(% at bottom-right = variance explained by top 3 PCs)",
        fontsize=9, y=0.993,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.985))
    fname_key = key.replace(" ", "_")
    out = NB_OUT_DIR / f"sec7_7_per_token_{fname_key}.png"
    fig.savefig(out, dpi=130)
    plt.show()
    return out


print(f"rendering §7.7 PCA-RGB per-token grids: {len(SEC77C_TARGET_TOKENS)} tokens")
for label, key in SEC77C_TARGET_TOKENS:
    print(f"  '{key}' ...")
    render_pca_rgb_per_token(label, key)


---
### 7.8 各 layer の **全 transformer blocks** を pool して PCA-RGB (per-token grid)

§7.7 では各 attention layer の `.t0` 1 個 (20 heads) しか見ていなかった。**本 §7.8 で全 t-block を pool した版を実装**する。

§7.7 では `.t0` だけ見ていたが、

> **attention 方向の復習** (§7 を読んでない読者向け): cross-attention の attention weight は **画像 (Q) → 単語 (K/V)** 向き。本 §7.8 の PCA-RGB ヒートマップは「**画像のどの位置がその単語に注目しているか**」を可視化している (情報自体は V 経由で 単語 → 画像 に流れる、attention の重み方向 画像→単語 とは別軸の話)。ここでは **同じ layer 内の全 `.t0..tN` を縦に concat** して **layer 全体 = 数十〜200 head の空間** で PCA を取り直す。

各 layer の合計 head 数:

| layer | t-blocks | heads/block | total heads |
|---|---:|---:|---:|
| down1.a0, .a1 | 2 | 10 | **20 each** |
| down2.a0, .a1 | 10 | 20 | **200 each** |
| mid.a0 | 10 | 20 | **200** |
| up0.a0, .a1, .a2 | 10 | 20 | **200 each** |
| up1.a0, .a1, .a2 | 2 | 10 | **20 each** |

**レイアウト (§7.6 style)**:
- **1 PNG / token** (8 tokens & phrases) — 「1 枚で、その token について全 11 layers × 全 head の挙動を一覧」
- 各 PNG: rows = 5 capture steps, cols = 11 U-Net layers
- 各 cell = pooled-heads PCA-RGB (sign 補正 + sqrt(ref) brightness)、右下に top3 PC 寄与率

実装:
1. **全 70 cross-attn modules を全 5 capture_steps で 1 回 capture** (1 forward pass で flag toggle、~60-90s)
2. (stage, stage_index, attn_block_index) で grouping (= 11 layer)
3. 各 group で全 t-blocks の (heads, Q) を縦 concat → (total_heads, Q)
4. §7.7 と同じ PCA-RGB 処理 (sign 補正 + sqrt(ref) brightness modulation)
5. **token 軸で 8 figures を render**: 各 figure = 11 layers × 5 steps grid

出力: `sec7_8_per_token_<key>.png` × 8 (token: witch / hair / staff / forest / mushrooms / magic_staff / enchanted_forest / bioluminescent_mushrooms)


---
#### §7.6 / §7.7 vs §7.8 の関係

| 観点 | §7.6 / §7.7 | §7.8 |
|---|---|---|
| 取り出す範囲 | 各 layer の `.t0` 1 個 (= 計 11 modules) | **全 70 modules** (各 layer の `.t0..tN` 全部) |
| 1 layer の head 集計 | head 平均 (e.g., 20 heads → 1 map) | **全 t-blocks の head 縦 concat** (e.g., down2.a0 は 10 × 20 = 200 heads → 1 map via PCA) |
| 観察できること | layer 単位の代表挙動 | **layer 全体の "深さ方向の冗長性"** ← top-3 PC 寄与率の高さで定量化 |

つまり §7.8 は **「`.t0` だけで近似できているのか?」** に直接答える設計。実測 (§7.8 出力) で寄与率が高ければ「冗長性あり、`.t0` 平均は妥当な approximation」、低ければ「t-block ごとに head 役割が違う、`.t0` は不十分な代表」と結論できる。

(U-Net 内 cross-attention の処理順 / layer 別 head 数の詳細は §7.2 で既出、ここでは省略)


In [ ]:
# 全 70 cross-attn modules を全 capture_steps ([0, 7, 14, 22, 29]) で capture
# 1 回の forward pass で全 step まとめ撮り (capture flag を step list で toggle)
# RecordingAttnProcessor は §7.3 で定義したものをそのまま再利用

capture_flag_all = {"on": False}
buf_all: list[dict] = []
orig_all: dict = {}

all_cross_names = {info["name"] for info in all_cross}
for name, mod in pipe.unet.named_modules():
    if isinstance(mod, Attention):
        orig_all[name] = mod.processor
        if name in all_cross_names:
            mod.processor = RecordingAttnProcessor(
                name, parse_module_info(name), capture_flag_all, buf_all)
        else:
            mod.processor = AttnProcessor()

print(f"swap: {len(orig_all)} attention modules, {len(all_cross_names)} を Recording に")

# 全 capture_steps で record
steps_to_capture_all = sorted(step_captured.keys())  # = [0, 7, 14, 22, 29]
print(f"capturing all {len(all_cross_names)} modules at steps={steps_to_capture_all}...")

pipe.scheduler.set_timesteps(num_inference_steps, device=device)
timesteps = pipe.scheduler.timesteps
gen_device = "cpu" if device == "mps" else device  # winport fix(B-1): randn と generator の device を揃える (cuda 必須)
gen_all = torch.Generator(device=gen_device).manual_seed(seed)
latents = torch.randn(latent_shape, generator=gen_all, dtype=latent_dtype, device=gen_device).to(device)
latents = latents * pipe.scheduler.init_noise_sigma

all_captured_per_step: dict[int, list[dict]] = {}
decoded_at_capture: dict[int, object] = {}  # PIL.Image per capture step
t0 = time.perf_counter()
for i, t in enumerate(timesteps):
    latent_model_input = torch.cat([latents] * 2) if do_cfg else latents
    latent_model_input = pipe.scheduler.scale_model_input(latent_model_input, t)
    capture_flag_all["on"] = (i in steps_to_capture_all)
    buf_all.clear()
    with torch.no_grad():
        noise_pred = unet(
            latent_model_input, t,
            encoder_hidden_states=prompt_embeds_in,
            added_cond_kwargs=added_cond_kwargs,
            return_dict=False,
        )[0]
    if do_cfg:
        np_u, np_t = noise_pred.chunk(2)
        noise_pred = np_u + guidance_scale * (np_t - np_u)
    latents = pipe.scheduler.step(noise_pred, t, latents, return_dict=False)[0]
    if i in steps_to_capture_all and buf_all:
        all_captured_per_step[i] = [dict(c) for c in buf_all]
        decoded_at_capture[i] = decode_latents_to_pil(latents)
        print(f"  step {i:3d}: captured {len(buf_all)} modules + decoded")

for name, mod in pipe.unet.named_modules():
    if isinstance(mod, Attention) and name in orig_all:
        mod.processor = orig_all[name]
print(f"capture done in {time.perf_counter() - t0:.1f}s, restored {len(orig_all)} processors")

# group by layer per step: groups_full_per_step[step][(stage, sIdx, aIdx)] = list of t-block recs
groups_full_per_step: dict[int, dict[tuple, list[dict]]] = {}
for step_idx, recs in all_captured_per_step.items():
    g: dict[tuple, list[dict]] = {}
    for rec in recs:
        info = rec["info"]
        k = (info["stage"], info["stage_index"], info["attn_block_index"])
        g.setdefault(k, []).append(rec)
    for k in g:
        g[k].sort(key=lambda r: r["info"]["transformer_index"] or 0)
    groups_full_per_step[step_idx] = g

# print summary using the last step
last_groups = groups_full_per_step[max(groups_full_per_step.keys())]
print(f"\ngrouped into {len(last_groups)} attention layers:")
for k in sorted(last_groups.keys(), key=lambda x: (STAGE_ORDER.get(x[0], 99), x[1] if x[1] is not None else 99, x[2] if x[2] is not None else 99)):
    recs = last_groups[k]
    n_t = len(recs)
    heads_per = recs[0]["heads"]
    Q = recs[0]["probs"].shape[1]
    side = int(round(Q ** 0.5))
    print(f"  {k}: {n_t} t-blocks × {heads_per} heads = {n_t * heads_per:3d} total heads ({side}×{side})")


In [ ]:
import numpy as np  # ensure numpy is available (recovery for kernels missing earlier import)
# 全 t-blocks を pool して PCA-RGB、§7.6 style (1 PNG / token、rows=steps, cols=layers)
from matplotlib import cm
from PIL import Image

def pca_rgb_for_group_token(groups_dict: dict, group_key: tuple, token_key: str):
    # 戻り値: (rgb (H, W, 3) in [0,1], top3 var ratio, total_heads) または (None, None, 0)
    recs = groups_dict.get(group_key, [])
    if not recs:
        return None, None, 0
    idxs = ttab["token_index"].get(token_key) or ttab["phrase_index"].get(token_key) or []
    if not idxs:
        return None, None, 0
    p_token_per_block = []
    for rec in recs:
        probs = rec["probs"]
        heads = rec["heads"]
        nb_ = probs.shape[0] // heads
        p = probs.view(nb_, heads, probs.shape[1], probs.shape[2])
        if nb_ == 2:
            p = p[1:2]
        p_token = p[0, :, :, idxs].sum(dim=2)
        p_token_per_block.append(p_token)
    p_all = torch.cat(p_token_per_block, dim=0)
    total_heads = p_all.shape[0]
    side = int(round(p_all.shape[1] ** 0.5))
    if side * side != p_all.shape[1]:
        return None, None, total_heads
    ref = p_all.mean(dim=0).numpy().astype(np.float32)
    X = p_all.numpy().T.astype(np.float32)
    Xc = X - X.mean(axis=0, keepdims=True)
    try:
        _, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    except np.linalg.LinAlgError:
        return None, None, total_heads
    n_pc = min(3, Vt.shape[0])
    Y = Xc @ Vt[:n_pc].T
    ref_c = ref - ref.mean()
    for k in range(n_pc):
        if float(np.dot(Y[:, k], ref_c)) < 0:
            Y[:, k] = -Y[:, k]
    if n_pc < 3:
        Y_full = np.zeros((Y.shape[0], 3), dtype=np.float32)
        Y_full[:, :n_pc] = Y
        Y = Y_full
    Y_min = Y.min(axis=0)
    Y_max = Y.max(axis=0)
    Y_norm = (Y - Y_min) / (Y_max - Y_min + 1e-9)
    ref_norm = (ref - ref.min()) / (ref.max() - ref.min() + 1e-9)
    rgb = Y_norm.reshape(side, side, 3) * np.sqrt(ref_norm).reshape(side, side, 1)
    var_total = float((S ** 2).sum())
    var_top3 = float((S[:n_pc] ** 2).sum())
    var_ratio = var_top3 / max(var_total, 1e-9)
    return rgb, var_ratio, total_heads


def render_pca_rgb_per_token(label: str, key: str) -> Path:
    # 1 figure / token: rows = capture steps (5), cols = 11 U-Net layers + 1 decoded image
    steps_avail = sorted(groups_full_per_step.keys())
    # layer (group) keys are consistent across steps — pick from any step
    sample_step = steps_avail[0]
    group_keys = sorted(groups_full_per_step[sample_step].keys(),
                        key=lambda k: (STAGE_ORDER.get(k[0], 99),
                                       k[1] if k[1] is not None else 99,
                                       k[2] if k[2] is not None else 99))
    n_modules = len(group_keys)
    n_cols = n_modules + 1  # +1 for decoded image column
    n_steps = len(steps_avail)
    fig, axes = plt.subplots(n_steps, n_cols,
                             figsize=(n_cols * 1.6, n_steps * 1.6))
    axes = np.atleast_2d(axes)

    for ri, step_idx in enumerate(steps_avail):
        groups_step = groups_full_per_step[step_idx]
        for cj, gk in enumerate(group_keys):
            ax = axes[ri, cj]
            ax.axis("off")
            rgb, var3, total_heads = pca_rgb_for_group_token(groups_step, gk, key)
            if rgb is None:
                ax.set_facecolor("#222")
                continue
            # 関数の契約: rgb が None でなければ var3 も必ず float (両方とも 1 失敗パスで None)。
            # pyright は要素ごと独立に Optional 扱いなので、ここで明示的に narrow する。
            assert var3 is not None
            rgb_u8 = (rgb * 255).astype(np.uint8)
            pil = Image.fromarray(rgb_u8).resize((128, 128), Image.Resampling.NEAREST)
            ax.imshow(pil)
            if ri == 0:
                recs = groups_step[gk]
                info0 = recs[0]["info"]
                short = module_short_name(info0).rsplit(".", 1)[0]
                Q = recs[0]["probs"].shape[1]
                side = int(round(Q ** 0.5))
                ax.set_title(f"{short}\n({side}×{side})\nheads={total_heads}",
                             fontsize=7)
            if cj == 0:
                ax.text(-0.20, 0.5, f"step {step_idx}",
                        transform=ax.transAxes, fontsize=8,
                        ha="right", va="center")
            ax.text(0.97, 0.02, f"{var3*100:.1f}%",
                    transform=ax.transAxes, fontsize=6,
                    ha="right", va="bottom", color="white",
                    bbox=dict(facecolor="black", alpha=0.55, pad=0.8,
                              edgecolor="none"))
        # last column: decoded image at this step
        ax_im = axes[ri, -1]
        ax_im.axis("off")
        img_dec = decoded_at_capture.get(step_idx)
        if img_dec is not None:
            ax_im.imshow(img_dec)
        if ri == 0:
            ax_im.set_title("decoded\nat step", fontsize=7)

    fig.suptitle(
        f"token = '{label}' — all 11 U-Net layers × 5 capture steps  "
        f"(PCA-RGB pooled per layer, % at bottom-right = variance explained by top 3 PCs, rightmost col = decoded image)",
        fontsize=9, y=0.993,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.985))
    fname_key = key.replace(" ", "_")
    out = NB_OUT_DIR / f"sec7_8_per_token_{fname_key}.png"
    fig.savefig(out, dpi=130)
    plt.show()
    return out


# render for all 8 tokens & phrases (row_specs から)
print(f"rendering per-token PCA-RGB grids: {len(row_specs)} figures...")
for label, key in row_specs:
    print(f"  rendering '{key}' ...")
    render_pca_rgb_per_token(label, key)
print(f"saved {len(row_specs)} PNGs to {NB_OUT_DIR}/")


<a id="head_profile"></a>

---
### 7.9 profile-based head finding — t0 の attention 役割を 1 ベクトルで要約

本節は **2 stage** の組み立てで、§7 冒頭で整理した「forward (画像→単語)」と「reverse (単語→画像)」の **両方向を経由** します。

**Stage 1 (forward の集約)**: 各 cross-attention t0 module について attention 行列 `P_havg ∈ R^{Q × 77}` (head 平均後) の **Q 軸 (image 位置) を平均** して 1 本の **profile** `∈ R^77` を得る。これは「全 image 位置が token を見ている **自然な向き (画像→単語)** の attention を **Q 軸で marginalize** した平均値」 — つまり **forward direction の情報を 1 本に集約** したもの:

```
profile[k] = (1/Q) · Σ_q P_havg[q, k]
           = この t0 が image 位置によらず平均的に token k をどれだけ見るか
```

- softmax の per-row 性質より、各 row が sum=1 → 平均後の profile も **sum=1 の確率分布**。module 間で直接比較可
- §7.11 で視覚化される「同 module の row は image 位置を変えても envelope が似ている」現象を 1 ベクトルで凝縮した表現でもある

**Stage 2 (profile からの逆解析: token → head 逆引き)**: profile を得た後で、target word `["witch", "staff", "mushrooms"]` の subtoken 位置の profile 値を「**この t0 がその単語をどれだけ強く attend するか**」のスコアに用い、各 word の **top-3 t0** を取る。これは「**単語 → head**」方向の逆引き = **逆解析の一種**。§7.5-7.8 の「単語 → 画像位置」逆解析と同じ系譜で、対象が「画像位置」から「head 集合」に置き換わっただけ。

**手順** (追加 forward 0、§7.4 capture から抽出のみ):

1. (Stage 1) 全 captured t0 について profile を計算
2. (Stage 2) target word の subtoken 位置の profile 合計値で rank、各 word の **top-3 t0** を選出
3. 各 word top-1 を最優先確保 (Stage A) → 各 word の top-2, top-3 を rank-major で追加 (Stage B) → 残り slot は全 word 合計 score で fallback 補充 (Stage C)、合計 **最大 10 個** の interesting t0 を確定
4. 選別 t0 の profile を colored strip 形式 (§7.11 と同じ format) で表示。stacked layout、行 = module、column = token position (padding cut)

**§7.10/§7.11 への接続**: 本節で確定した **各 RANK_WORD の top-1 t0** (= `top_per_word[word][0]`) を、次の 2 節が直接利用する:

- **§7.10 (self-attention probe)**: 各 word の top-1 t0 module 内で cross-attn argmax の image 位置を query にし、同じ transformer block の sibling self-attn module (`.attn1`) で key 分布を可視化 (3 panel)
- **§7.11 (cross-attention の 画像→単語 view)**: §7.10 と **同じ (word, head, query) の 3 panel** で、今度は cross-attn 行 (image 位置 → 77 token 分布) を見る。§7.10 と §7.11 は同じ triplet の **dual view** (§7.10 = image 側 2D 分布、§7.11 = token 側 1D 分布)

出力: `sec7_9_t0_profiles.png` (1 図、選別 10 module の colored strip stacked layout)。


In [ ]:
# §7.9: profile-based head (t0-level) finding
#   各 t0 の attention 行列 (head 平均後) を image-位置軸で平均 → 1 本の (77,) profile
#   target word への profile mass で rank、union で最大 10 個の "interesting t0" を選別
# 追加 forward 0 (§7.4 capture から抽出のみ)。

# EOS 境界 (§7.11 でも使うが、本節が先に走るのでここで定義)
EOS_TOKEN_ID = pipe.tokenizer.eos_token_id
assert EOS_TOKEN_ID is not None
ids_77 = ttab["ids"]
first_eos = next((i for i, x in enumerate(ids_77) if x == EOS_TOKEN_ID and i > 0), len(ids_77))
strip_end = first_eos + 1
print(f"first_eos = {first_eos}  (strip 表示範囲 [0, {strip_end}))")

# 1. 全 captured t0 の profile を計算
t0_profiles: dict[str, np.ndarray] = {}
for rec in step_captured[last_step]:
    probs = rec["probs"]                # (B*heads, Q, K=77)
    heads_n = rec["heads"]
    n_batch = probs.shape[0] // heads_n
    pv = probs.view(n_batch, heads_n, probs.shape[1], probs.shape[2])
    if n_batch == 2:
        pv = pv[1:2]                    # CFG text side
    p_havg = pv.mean(dim=1)[0]          # (Q, 77) head 平均
    profile = p_havg.mean(dim=0)        # (77,) image-位置平均
    t0_profiles[rec["name"]] = profile.cpu().numpy()

print(f"\nprofiles computed for {len(t0_profiles)} t0 modules (last_step={last_step})")
sum_check = float(next(iter(t0_profiles.values())).sum())
print(f"sum check: 1 profile の sum = {sum_check:.4f}  (softmax の per-row 性質 → 平均後も sum=1)")

# 2. target word への profile mass で rank
RANK_WORDS = ["witch", "staff", "mushrooms"]
TOP_N = 3
scores: dict[tuple[str, str], float] = {}
for name, prof in t0_profiles.items():
    for word in RANK_WORDS:
        idxs = ttab["token_index"].get(word, [])
        if idxs:
            scores[(name, word)] = float(prof[idxs].sum())

top_per_word: dict[str, list[str]] = {}
print(f"\n--- top-{TOP_N} t0 per target word ---")
for word in RANK_WORDS:
    ranked = sorted(t0_profiles.keys(),
                    key=lambda n: scores.get((n, word), 0.0),
                    reverse=True)
    top_per_word[word] = ranked[:TOP_N]
    print(f"  {word!r}:")
    for r, name in enumerate(top_per_word[word], 1):
        print(f"    {r}. {name:24s} profile mass = {scores[(name, word)]:.4f}")

# 3. selected_t0_by_profile を 3 stage で構築
#   Stage A: 各 word の top-1 を最優先で確定 (必ず含める)
#   Stage B: top-2, top-3 を rank-major (まず全 word の top-2、次に全 word の top-3)
#   Stage C: 残り slot を全 word 合計 score で埋める
MAX_INTERESTING = 10
selected_t0_by_profile: list[str] = []
seen: set[str] = set()

# Stage A: 各 word top-1 (必ず確保)
for word in RANK_WORDS:
    if top_per_word[word]:
        name = top_per_word[word][0]
        if name not in seen:
            selected_t0_by_profile.append(name)
            seen.add(name)

# Stage B: rank-major で top-2, top-3 ... を埋める (全 word の top-2 → 全 word の top-3 ...)
for rank_idx in range(1, TOP_N):
    for word in RANK_WORDS:
        if len(top_per_word[word]) <= rank_idx:
            continue
        name = top_per_word[word][rank_idx]
        if name not in seen and len(selected_t0_by_profile) < MAX_INTERESTING:
            selected_t0_by_profile.append(name)
            seen.add(name)

# Stage C: 残り slot を全 word 合計 score で fallback 補充
if len(selected_t0_by_profile) < MAX_INTERESTING:
    fallback = sorted(
        [n for n in t0_profiles if n not in seen],
        key=lambda n: sum(scores.get((n, w), 0.0) for w in RANK_WORDS),
        reverse=True)
    for n in fallback:
        if len(selected_t0_by_profile) >= MAX_INTERESTING:
            break
        selected_t0_by_profile.append(n)
        seen.add(n)

# total score の定義: profile の RANK_WORDS subtoken 位置の値の合計 (= profile[witch] + profile[staff] + profile[mushrooms])
# Stage C の fallback ranking はこの total score 順
def _total_score(name: str) -> float:
    return sum(scores.get((name, w), 0.0) for w in RANK_WORDS)

print(f"\n--- selected {len(selected_t0_by_profile)} interesting t0 (stage A: top-1 / B: top-2,3 / C: fallback) ---")
print(f"    (total = sum of profile mass on {RANK_WORDS}'s subtoken positions)")
for i, name in enumerate(selected_t0_by_profile):
    tags = []
    for w in RANK_WORDS:
        if name in top_per_word[w]:
            tags.append(f"{w}#{top_per_word[w].index(name) + 1}")
    total = _total_score(name)
    rank_note = " | ".join(tags) if tags else "(fallback)"
    print(f"  [{i:2d}] {name:24s}  total={total:.4f}  -> {rank_note}")

# 参考: fallback 候補 (selected に未採用な module) の total score 一覧
remaining = [n for n in t0_profiles if n not in seen]
if remaining:
    print(f"\n--- remaining (unselected) t0 sorted by total score ---")
    for n in sorted(remaining, key=_total_score, reverse=True):
        print(f"  {n:24s}  total={_total_score(n):.4f}")

# 4. 可視化: stacked colored strips (§7.11 Figure A と同じ strip 形式の stacked 版)
n_show = len(selected_t0_by_profile)
fig, axes = plt.subplots(n_show, 1, figsize=(11, n_show * 0.55 + 1.2),
                          constrained_layout=True)
axes = np.atleast_1d(axes)
cmap = plt.get_cmap("inferno")

for ri, name in enumerate(selected_t0_by_profile):
    ax = axes[ri]
    profile = t0_profiles[name]
    vmax = max(float(profile[:strip_end].max()), 1e-9)
    ax.set_xlim(-0.5, strip_end - 0.5)
    ax.set_ylim(0, 1)
    ax.set_yticks([])
    ax.set_xticks([])
    for i in range(strip_end):
        v = float(profile[i]) / vmax
        ax.axvspan(i - 0.5, i + 0.5, color=cmap(v))
        # token text は全 row に表示 (§7.11 と統一)。clip_on=True で長い token が
        # 隣 row へオーバーフローして constrained_layout に axis を圧縮されるのを防ぐ
        # (startoftext / masterpiece / endoftext などは端が clip される)
        t = ttab["tokens"][i].replace("</w>", "")
        text_color = "white" if v < 0.55 else "black"
        ax.text(i, 0.5, t, ha="center", va="center",
                fontsize=7, color=text_color, rotation=90, clip_on=True)
    tags = []
    for w in RANK_WORDS:
        if name in top_per_word[w]:
            tags.append(f"{w}#{top_per_word[w].index(name) + 1}")
    ylab = f"{name}\n{' '.join(tags)}" if tags else name
    ax.set_ylabel(ylab, fontsize=7, rotation=0, ha="right", va="center")

axes[0].set_title(
    f"§7.9 t0-level profile (mean over Q) — {n_show} interesting modules\n"
    f"(rank by attention to: {', '.join(repr(w) for w in RANK_WORDS)}, "
    f"last_step={last_step}, padding cut to first_eos={first_eos})",
    fontsize=10)

out_path = NB_OUT_DIR / "sec7_9_t0_profiles.png"
fig.savefig(out_path, dpi=110)
print(f"\nsaved: {out_path}")
plt.show()


**観察ポイント — §7.9 profile から見えること** (簡易)

10 row の選別は **各 RANK_WORD の top-1 を必ず確保した上で残りを補充** (Stage A → B → C) という構成的な選び方なので、target word 位置 (`witch`=k=3 / `staff`=k=9 / `mushrooms`=k=22) に明るい cell が立つ row が並ぶのは「狙ったものが出ている」確認以上の意味はない (= selection の sanity)。

それ以外に図から read 取れる baseline 傾向 (副次的):

- **k=0 (BOS) と k=1 (不定冠詞 `a`)** がほぼ全 row で明るい — head 役割と独立な CLIP encoder 構造由来の baseline (一般に "attention sink" として知られる前方トークン集中現象と整合)
- **k=3 (`witch`)** は staff / mushrooms top-1 module でも比較的明るい — 主役 token の broadcast 傾向 (この prompt × seed の anecdotal、§7.11 観察 2 で「witch dominance」として再登場)
- **k=32 (EOS)** は module 次第で明暗が混在 — 文末意味保持の head 間不均一

本節は §7.10 / §7.11 の 3 panel triplet を確定する **head 選別ステップ** であり、profile 自体の体系的解釈は本ノートのスコープ外。


---
### 7.10 self-attention probe — §7.9 で選ばれた top-1 head での 3 panel sample

self-attention の分布を **§7.9 で identify した interesting head の中で、各 RANK_WORD の top-1 (= その語に最も強く反応する head)** に絞って観察する。3 panel で構成:

| panel | word | self-attn module (cross-attn `.attn2` の sibling `.attn1`) | query 位置 |
|---|---|---|---|
| 1 | witch | `up_blocks.1.attentions.0.t0.attn1` | cross-attn argmax of `witch` in same module |
| 2 | staff | `up_blocks.0.attentions.0.t0.attn1` | cross-attn argmax of `staff` in same module |
| 3 | mushrooms | `up_blocks.0.attentions.0.t0.attn1` (= panel 2 と同 module、query 位置のみ別) | cross-attn argmax of `mushrooms` |

実際の head 名は §7.9 の `top_per_word[word][0]` から動的に取り、`.attn2` → `.attn1` で sibling self-attn module を pair する。staff#1 と mushrooms#1 が同 module なので **unique self-attn module は 2 個** で済む。

**実行コスト**: 2 unique self-attn module を 1 forward pass で同時 capture (Apple Silicon で 30〜60 秒)。


In [ ]:
from diffusers.models.attention_processor import AttnProcessor

# RecordingSelfAttnProcessor (cross-attn 用 RecordingAttnProcessor と同形、is_cross 判定だけ反転)

class RecordingSelfAttnProcessor(AttnProcessor):
    # self-attention の attention_probs を capture_now_flag が ON のときだけ保存
    # AttnProcessor を継承する (RecordingAttnProcessor と同様、attn.processor 代入の nominal 型一致用)。

    def __init__(self, name: str, info: dict, capture_now_flag: dict, captured: list):
        super().__init__()   # AttnProcessor は __init__ なしなので no-op
        self.name = name
        self.info = info
        self.capture_now_flag = capture_now_flag
        self.captured = captured

    def __call__(self, attn, hidden_states, encoder_hidden_states=None,
                 attention_mask=None, temb=None, *args, **kwargs):
        # SD1.5 / SDXL の通常 self/cross-attention では to_k / to_v / to_out は必ず存在する。
        # Pyright 用に narrowing しておく (Diffusers の Attention は joint cross attention
        # 等を見越して to_k / to_v / to_out を Optional 型で保持している)。
        assert attn.to_k is not None and attn.to_v is not None and attn.to_out is not None
        residual = hidden_states
        input_ndim = hidden_states.ndim
        if input_ndim == 4:
            batch_size, channel, h, w = hidden_states.shape
            hidden_states = hidden_states.view(batch_size, channel, h * w).transpose(1, 2)
        batch_size, seq_len, _ = (
            hidden_states.shape if encoder_hidden_states is None else encoder_hidden_states.shape
        )
        attention_mask = attn.prepare_attention_mask(attention_mask, seq_len, batch_size)  # pyright: ignore[reportArgumentType]
        if attn.group_norm is not None:
            hidden_states = attn.group_norm(hidden_states.transpose(1, 2)).transpose(1, 2)
        query = attn.to_q(hidden_states)
        is_cross = encoder_hidden_states is not None
        if encoder_hidden_states is None:
            encoder_hidden_states = hidden_states
        elif attn.norm_cross:
            encoder_hidden_states = attn.norm_encoder_hidden_states(encoder_hidden_states)
        key = attn.to_k(encoder_hidden_states)
        value = attn.to_v(encoder_hidden_states)
        query = attn.head_to_batch_dim(query)
        key = attn.head_to_batch_dim(key)
        value = attn.head_to_batch_dim(value)
        attention_probs = attn.get_attention_scores(query, key, attention_mask)

        if (not is_cross) and self.capture_now_flag.get("on", False):
            self.captured.append({
                "name": self.name,
                "info": self.info,
                "shape": list(attention_probs.shape),
                "probs": attention_probs.detach().to(torch.float32).cpu().clone(),
                "heads": int(attn.heads),
                "batch": int(batch_size),
                "query_len": int(attention_probs.shape[1]),
                "key_len":   int(attention_probs.shape[2]),
            })

        hidden_states = torch.bmm(attention_probs, value)
        hidden_states = attn.batch_to_head_dim(hidden_states)
        hidden_states = attn.to_out[0](hidden_states)
        hidden_states = attn.to_out[1](hidden_states)
        if input_ndim == 4:
            hidden_states = hidden_states.transpose(-1, -2).reshape(batch_size, channel, h, w)
        if attn.residual_connection:
            hidden_states = hidden_states + residual
        hidden_states = hidden_states / attn.rescale_output_factor
        return hidden_states


# §7.9 で選ばれた各 RANK_WORD の top-1 cross-attn head から、3 panel 構成 (word, self-attn, cross-attn) を組む
# - self-attn (attn1) と cross-attn (attn2) は同じ transformer_block の兄弟ペア (path は ".attn2" → ".attn1" で導出)
# - 各 panel = (word, その word の top-1 head)。staff#1 と mushrooms#1 が同 module なら self-attn は重複 (2 unique self-attn modules を capture すれば足りる)
SELF_ATTN_PANELS: list[tuple[str, str, str]] = []  # (word, self_name, cross_name)
for word in RANK_WORDS:
    cross_name = top_per_word[word][0]
    self_name = cross_name.replace(".attn2", ".attn1")
    SELF_ATTN_PANELS.append((word, self_name, cross_name))

unet_module_names = {n for n, _ in pipe.unet.named_modules()}
for word, self_name, cross_name in SELF_ATTN_PANELS:
    assert self_name in unet_module_names,  f"missing self-attn: {self_name}"
    assert cross_name in unet_module_names, f"missing cross-attn: {cross_name}"

print(f"{len(SELF_ATTN_PANELS)} panels (RANK_WORDS × top-1 cross-attn head):")
for word, self_name, cross_name in SELF_ATTN_PANELS:
    print(f"  word={word!r:12s}  self-attn @ {self_name.split('.', 1)[1]}")
unique_self_names_710 = sorted({sn for _, sn, _ in SELF_ATTN_PANELS})
print(f"unique self-attn modules to capture: {len(unique_self_names_710)} ({len(SELF_ATTN_PANELS) - len(unique_self_names_710)} duplicate skipped)")


In [ ]:
# capture_self_attns(module_names) — N 個の module を **1 forward pass で同時 capture**
# 1 module 用の per-call 実装は per-module で 30-step loop を N 回走るので無駄。
# まとめて 1 回の loop で済ませる (§7.4 cross-attn capture と同じ batch 方式)。
# 戻り値: dict[module_name → (head 平均後 (Q, K) matrix, side)]

def capture_self_attns(module_names: list[str], step_to_capture: int) -> dict[str, tuple[np.ndarray, int]]:
    flag = {"on": False}
    buf: list[dict] = []
    orig: dict = {}
    targets = set(module_names)
    for name, mod in pipe.unet.named_modules():
        if isinstance(mod, Attention):
            orig[name] = mod.processor
            if name in targets:
                mod.processor = RecordingSelfAttnProcessor(
                    name, parse_module_info(name), flag, buf)
            else:
                mod.processor = AttnProcessor()

    pipe.scheduler.set_timesteps(num_inference_steps, device=device)
    ts = pipe.scheduler.timesteps
    gen_device = "cpu" if device == "mps" else device  # winport fix(B-1): randn と generator の device を揃える (cuda 必須)
    gen = torch.Generator(device=gen_device).manual_seed(seed)
    lat = torch.randn(latent_shape, generator=gen, dtype=latent_dtype, device=gen_device).to(device)
    lat = lat * pipe.scheduler.init_noise_sigma

    captured_recs: dict[str, dict] = {}
    for i, t in enumerate(ts):
        lmi = torch.cat([lat] * 2) if do_cfg else lat
        lmi = pipe.scheduler.scale_model_input(lmi, t)
        flag["on"] = (i == step_to_capture)
        buf.clear()
        with torch.no_grad():
            noise = unet(lmi, t, encoder_hidden_states=prompt_embeds_in,
                         added_cond_kwargs=added_cond_kwargs, return_dict=False)[0]
        if do_cfg:
            n_u, n_t = noise.chunk(2)
            noise = n_u + guidance_scale * (n_t - n_u)
        lat = pipe.scheduler.step(noise, t, lat, return_dict=False)[0]
        if i == step_to_capture:
            for rec in buf:
                captured_recs[rec["name"]] = dict(rec)

    # restore processors
    for name, mod in pipe.unet.named_modules():
        if isinstance(mod, Attention) and name in orig:
            mod.processor = orig[name]

    # head 平均 + CFG text 側 → (Q, K)
    results: dict[str, tuple[np.ndarray, int]] = {}
    for name in module_names:
        rec = captured_recs.get(name)
        assert rec is not None, f"capture failed for {name}"
        probs = rec["probs"]            # (B*heads, Q, K)
        heads = rec["heads"]
        nb_ = probs.shape[0] // heads
        p = probs.view(nb_, heads, probs.shape[1], probs.shape[2])
        if nb_ == 2:
            p = p[1:2]
        mat = p.mean(dim=1)[0].numpy()
        side = int(round(mat.shape[0] ** 0.5))
        assert side * side == mat.shape[0]
        results[name] = (mat, side)
    return results


# §7.9 で選ばれた unique self-attn modules を 1 forward pass で同時 capture
self_final_step = num_inference_steps - 1
print(f"capturing self-attn at {len(unique_self_names_710)} unique modules in 1 forward pass (step i={self_final_step})...")
t0 = time.perf_counter()
captured_dict = capture_self_attns(unique_self_names_710, self_final_step)
print(f"done in {time.perf_counter() - t0:.1f}s")
print()

# self_data_by_name[self_name] = (mat, side)
self_data_by_name: dict[str, tuple[np.ndarray, int]] = {}
for sn in unique_self_names_710:
    mat, side = captured_dict[sn]
    self_data_by_name[sn] = (mat, side)
    print(f"  {sn.split('.', 1)[1]:50s}  matrix={mat.shape}  ({side}x{side})")


In [ ]:
# 各 panel ごとに word の cross-attn argmax で query 位置 (qy, qx) を決定 → self-attn 行列の row を抜く
self_panels_data: list[dict] = []
for word, self_name, cross_name in SELF_ATTN_PANELS:
    arr = heatmap_db.get((last_step, cross_name, word))
    assert arr is not None, f"missing cross-attn heatmap for ({cross_name}, {word})"
    qy, qx = np.unravel_index(int(arr.argmax()), arr.shape)
    val = float(arr[qy, qx])
    mat, side = self_data_by_name[self_name]
    cross_res = module_res_db.get(cross_name)
    assert cross_res == (side, side), \
        f"cross/self spatial mismatch: cross={cross_res}, self=({side},{side})"
    q_idx = int(qy) * side + int(qx)
    self_panels_data.append({
        "word": word,
        "self_name": self_name,
        "cross_name": cross_name,
        "qy": int(qy), "qx": int(qx), "val": val,
        "side": side,
        "key_map": mat[q_idx].reshape(side, side),
    })

print("self-attn key maps per panel (query = word's cross-attn argmax in word's top-1 head):")
for p in self_panels_data:
    short = p["self_name"].split(".", 1)[1]
    print(f"  word={p['word']!r:12s}  self-attn @ {short:50s}  "
          f"q=({p['qy']:2d},{p['qx']:2d}) cross-attn val={p['val']:.4f}  "
          f"key_map={p['key_map'].shape}")


In [ ]:
# 1 行 × 3 列 layout。各 panel = decoded 画像 (grayscale) + self-attn (gamma boost overlay) + query marker
from matplotlib import cm  # noqa: F401
import matplotlib as mpl

W, H = decoded[num_inference_steps - 1].size
decoded_final = decoded[num_inference_steps - 1]
gray_bg = np.array(decoded_final.convert("L")) / 255.0   # (H, W) in [0, 1]

def draw_query_marker(ax, mx, my):
    # White halo + cyan center で grayscale 上でもハッキリ視認可能
    ax.plot(mx, my, marker="+", color="white",
            markersize=24, markeredgewidth=4.0, zorder=10)
    ax.plot(mx, my, marker="+", color="cyan",
            markersize=20, markeredgewidth=2.2, zorder=11)

n_cols = len(self_panels_data)
fig, axes = plt.subplots(1, n_cols, figsize=(n_cols * 5.2, 5.8),
                         constrained_layout=True)
axes = np.atleast_1d(axes)

for ci, p in enumerate(self_panels_data):
    ax = axes[ci]
    ax.axis("off")
    side = p["side"]
    arr = p["key_map"]

    # 正規化 (per-panel) + α と color index の両方に強い gamma boost をかけて
    # weak attention まで見えるようにする。旧版 (α=sqrt(a)、color index は a そのまま) では
    # mushrooms panel の「他のキノコへの secondary 拡散」が inferno の暗紫 × 弱 α で
    # 潰れて見えなかった。背景 grayscale α は 0.40 据え置き、強調は heatmap 側に集約。
    a = arr.copy()
    if a.max() > a.min():
        a = (a - a.min()) / (a.max() - a.min())

    # α は power 0.10 (旧 sqrt = power 0.50 から大幅 boost — ほぼ全画素を opaque に。
    # attention の強弱は α ではなく色 (下の a_color) で読む形)
    a_emph = np.power(a, 0.10)
    # color index にも gamma 0.50 適用 → inferno の暗紫帯を warm 帯へシフト
    # (旧版は a そのままだったので低 attention は暗紫で grayscale 背景に紛れていた)
    a_color = np.power(a, 0.50)

    # upsample (低解像 attention map を画像サイズへ)
    heat_norm_up = np.asarray(
        Image.fromarray((a_color * 255).astype(np.uint8)).resize((W, H), Image.Resampling.BILINEAR)
    ) / 255.0
    heat_alpha_up = np.asarray(
        Image.fromarray((a_emph * 255).astype(np.uint8)).resize((W, H), Image.Resampling.BILINEAR)
    ) / 255.0
    heat_rgb = mpl.colormaps["inferno"](heat_norm_up)[..., :3]

    # 背景: grayscale decoded 画像 (α 0.40 据え置き、heatmap 側の gamma boost で勝負する)
    ax.imshow(gray_bg, cmap="gray", alpha=0.40, vmin=0, vmax=1)
    # 前景: attention map (per-pixel α = power 0.10 → 弱所も opaque、強弱は色で読む)
    ax.imshow(heat_rgb, alpha=heat_alpha_up)

    # query marker (image 座標へ換算)
    mx = (p["qx"] + 0.5) * W / side
    my = (p["qy"] + 0.5) * H / side
    draw_query_marker(ax, mx, my)

    short_self = p["self_name"].split(".", 1)[1].replace(".transformer_blocks.0.attn1", ".attn1")
    ax.set_title(
        f"'{p['word']}' — q=({p['qy']},{p['qx']}) [{side}×{side}]\n"
        f"self-attn @ {short_self}",
        fontsize=10)

fig.suptitle(
    "§7.10 self-attention key distribution + decoded image overlay\n"
    "query marker (+) = each word's cross-attn argmax → 位置が witch / staff / mushrooms の視覚的位置に一致するか確認",
    fontsize=11)
out_path = NB_OUT_DIR / "sec7_10_self_attn_grid.png"
fig.savefig(out_path, dpi=120)
print(f"saved: {out_path}")
plt.show()


**観察ポイント — query marker の visual 位置一致 + self-attn の near-diagonal 性**

**観察範囲の制約 (重要、本観察を読む前に)**:

- **3 panels のみ** (witch / staff / mushrooms の top-1 head に固定) ← SDXL Base には self-attn ~70 個
- 最終 step (i=29) 1 step のみ ← 全 30 steps
- head 平均後 (head 別 pattern を見ていない) ← module ごとに 10 or 20 heads
- t0 のみ ← 各 attention layer に 2 or 10 t-blocks
- 3 query 位置のみ ← module ごとに 1024 or 4096 query 位置

つまり **数十万 head-step-module 計算のうち 2 module × 1 step × (head 平均) × 3 query** という狭い slice。

### 観察 1: query marker (+) は word の視覚的位置に乗っている

各 panel の `+` は **そのword の cross-attn argmax の image 位置** に置いてある。decoded 画像を grayscale 背景に出したことで、marker が実際の被写体と一致するか目視可能:

- `'witch'` q=(12, 36) on **64×64**: marker は **魔女の顔/上半身** 付近に落ちる。cross-attn val = **0.2594** ← 3 panel 中で突出 (1 token に ~26% 集中)
- `'staff'` q=(9, 10) on **32×32**: marker は **杖の付近**。cross-attn val = **0.0765** (witch の ~1/3)
- `'mushrooms'` q=(10, 3) on **32×32**: marker は **キノコ群の付近**。cross-attn val = **0.0750**

→ 3 word とも cross-attn argmax は対応する visual 位置と **概ね一致** (この prompt + seed の例では sanity check 通過)。「cross-attention は token を 1 つの image 領域に localize する」前提が成立している証拠。peak 強度の witch ≫ staff ≈ mushrooms は §7.9 profile mass (0.0997 / 0.0397 / 0.0331) と整合的。

### 観察 2: self-attn は near-diagonal + 同 semantic object への長距離 reach

各 panel の inferno overlay = **その query 位置からの self-attn 行 (key 分布)** = 「query 位置が image の他のどこを見るか」を表す。

- **共通する近傍集中 (局所性)**: 3 panel どれも query 位置の **近傍 (+marker から数 cell 以内)** が最も濃く reach する。self-attn の local smoothing 性質はやはり存在する
- **mushrooms panel の long-range semantic reach**: それに加えて、mushrooms panel では query 位置 (1 つのキノコ上) から **画像内の他のキノコ群** にも明確に attention が伸びている。距離は image 上で数十 cell 単位 — **near-diagonal の局所 smoothing では説明できない長距離 reach** が同時に出ている。head が「同じ semantic class の object」を attend していると解釈できる
- witch / staff panel ではこの semantic reach は (この例では) 弱い。画像内に同種 object が複数なく、reach 先が存在しないため

→ **self-attn の役割は「3×3 conv の generalization (= 局所 smoothing のみ)」では捉えきれない**。局所 smoothing と、**semantic similarity を介した非局所 reach** が両立する。後者は cross-attn (テキスト経由の attention) とは別の機構で、image-internal な「同一 concept の空間 broadcast」を担う候補。

役割分担:

| | 役割 | 距離特性 (今回の観察) |
|---|---|---|
| **cross-attention** | token (テキスト) → 空間位置 の bridge | 1 image 位置に 1 token が結びつく (long-range なし、観察 1) |
| **self-attention** | 空間位置 → 空間位置 の relation | near-diagonal (局所 smoothing) + 同 semantic object への長距離 reach (mushrooms 例、観察 2) |

**留意点 (本観察の限界、caveats)**:

- 観察 1: 視覚位置一致は「**概ね一致**」レベル。pixel 単位の正確性は cross-attn map の解像度 (32×32 / 64×64) と decoded 画像 (1024×1024) の差で厳密には判定不可
- 観察 2: head 平均後の像なので、head 別では near-diagonal 寄りの head と semantic reach 寄りの head に分かれている可能性 (= 異質な head 役割が混ざって観察されている可能性)
- 別 step (early step では global、late step では local など) で挙動が変わる可能性は残る
- 別 t-block (`.t1..t9` 等) や別 module (down2, mid 等、本節では外した位置) でも検証する必要あり
- §7.9 で word-specific に選んだのは "cross-attn の世界での top-1"、self-attn 役割とは別物 — self-attn の "本当に興味深い head" を選ぶには別 ranking が要る
- 本観察は「query marker = 視覚位置 一致」や「mushrooms panel での semantic reach」を **一般則として証明するものではなく**、限定 sample での **観察例**


---
### 7.11 cross-attention の 画像→単語 view — §7.9 top-1 head × RANK_WORDS の 3 panel

ここまでの §7 は **「token を固定 → 画像 2D 分布を見る」** という方向 (= attention の逆解析、§7.5-7.8) だった。本節は attention 行列の **dual view** = **「Q 軸の行を取って token 側 1D で見る」** (= attention 計算の自然な向きの read、画像 → 単語)。

§7.10 と **同じ 3 panel triplet** (RANK_WORDS × 各 word の top-1 cross-attn head) を使い、`SELF_ATTN_PANELS` から `(word, self_name, cross_name)` を引いて cross-attn 行列の row を抜く。§7.10 が self-attn の image 側 2D を見せたのに対し、§7.11 は **同じ query 位置からの cross-attn 行を 77 token の 1D 分布として** 見せる (両節は同じ 3 panel の dual view)。

**追加実行コスト 0**: §7.4 で capture 済の cross-attention rec と、§7.9 の `top_per_word` を再利用するだけ (新 forward なし)。

**可視化**: 2 図に分割。Layout は **1 行 × 3 列 = 3 panels** (§7.10 と対称):

- **Figure A** (`sec7_11a_token_strip.png`) — 各 panel は **token strip (padding cut)**: 実プロンプト token を BOS〜最初の EOS まで横並びに描画、背景色 = attention 値の colormap、originating token (query 決定に使った word) を緑枠で囲む
- **Figure B** (`sec7_11b_post_eos_leakage.png`) — 3 panel の 77-token 分布を **1 つのプロットに overlay**。EOS 境界を赤点線、padding 領域 (k=first_eos+1 〜 76) を薄く塗る。3 線でも「EOS 以降にも非ゼロ attention が乗る」現象は十分確認できる


In [ ]:
# §7.11: cross-attention の 画像→単語 view (§7.10 と同じ 3 panel triplet を使う)
#   Figure A : 1×3 grid の token strip (padding cut)
#   Figure B : 3 lines を 1 plot に overlay — EOS 以降の漏れ観察用
# 追加 forward 0 (§7.4 capture + §7.9 top_per_word + §7.10 SELF_ATTN_PANELS を再利用)。

# EOS 境界
EOS_TOKEN_ID = pipe.tokenizer.eos_token_id
assert EOS_TOKEN_ID is not None
ids_77 = ttab["ids"]
first_eos = next((i for i, x in enumerate(ids_77) if x == EOS_TOKEN_ID and i > 0), len(ids_77))
strip_end = first_eos + 1
print(f"first_eos = {first_eos}  (strip A 範囲 [0, {strip_end})、padding {76 - first_eos} 個)")

# 各 panel (word, self_name, cross_name) ごとに cross-attn から (77,) row を抜く
panels: list[dict] = []
for word, _self_name, cross_name in SELF_ATTN_PANELS:
    rec = next((r for r in step_captured[last_step] if r["name"] == cross_name), None)
    assert rec is not None, f"no cross-attn rec for {cross_name}"
    probs = rec["probs"]                # (B*heads, Q, K=77)
    heads_n = rec["heads"]
    n_batch = probs.shape[0] // heads_n
    pv = probs.view(n_batch, heads_n, probs.shape[1], probs.shape[2])
    if n_batch == 2:
        pv = pv[1:2]                    # CFG text side
    p_avg = pv.mean(dim=1)[0]           # (Q, 77) head 平均

    # cross-attn argmax で query 位置を決定
    arr = heatmap_db.get((last_step, cross_name, word))
    assert arr is not None
    qy, qx = np.unravel_index(int(arr.argmax()), arr.shape)
    side = int(round(p_avg.shape[0] ** 0.5))
    q_idx = int(qy) * side + int(qx)
    panels.append({
        "word": word,
        "cross_name": cross_name,
        "qy": int(qy), "qx": int(qx), "side": side,
        "dist_77": p_avg[q_idx].cpu().numpy(),
        "orig_idxs": ttab["token_index"].get(word, []),
    })

print(f"panels: {len(panels)}  (= {len(SELF_ATTN_PANELS)} = RANK_WORDS × top-1)")
for p in panels:
    short = p["cross_name"].split(".", 1)[1].replace(".transformer_blocks.0.attn2", ".attn2")
    print(f"  word={p['word']!r:12s}  cross-attn @ {short:45s}  q=({p['qy']:2d},{p['qx']:2d})  side={p['side']}")

# ==================================================================
# Figure A: 3×1 vertical stacking (each strip wider for readability)
# ==================================================================
n_panels = len(panels)
fig_a, axes_a = plt.subplots(n_panels, 1, figsize=(13, n_panels * 1.9 + 1.0),
                              constrained_layout=True)
axes_a = np.atleast_1d(axes_a)
cmap = plt.get_cmap("inferno")

for ri, p in enumerate(panels):
    ax = axes_a[ri]
    dist = p["dist_77"]
    vmax = max(float(dist[:strip_end].max()), 1e-9)
    ax.set_xlim(-0.5, strip_end - 0.5)
    ax.set_ylim(0, 1)
    ax.set_xticks([])
    ax.set_yticks([])
    for i in range(strip_end):
        v = float(dist[i]) / vmax
        ax.axvspan(i - 0.5, i + 0.5, color=cmap(v))
        t = ttab["tokens"][i].replace("</w>", "")
        text_color = "white" if v < 0.55 else "black"
        ax.text(i, 0.5, t, ha="center", va="center",
                fontsize=8, color=text_color, rotation=90)
    for idx in p["orig_idxs"]:
        if idx < strip_end:
            ax.add_patch(plt.Rectangle(
                (idx - 0.5, 0.02), 1, 0.96, fill=False,
                edgecolor="lime", linewidth=2.2, zorder=10))
    short = p["cross_name"].split(".", 1)[1].replace(".transformer_blocks.0.attn2", ".attn2")
    ax.set_title(
        f"word='{p['word']}' (={p['word']}#1)\n"
        f"cross-attn @ {short}\nq=({p['qy']},{p['qx']})",
        fontsize=9)

fig_a.suptitle(
    "§7.11A token strip (3-panel vertical stack) — image 位置 → 77 token 分布 "
    "(RANK_WORDS × top-1 head, padding cut, 緑枠=originating word)",
    fontsize=11)
out_a = NB_OUT_DIR / "sec7_11a_token_strip.png"
fig_a.savefig(out_a, dpi=130)
print(f"saved: {out_a}")
plt.show()

# ==================================================================
# Figure B: 3-panel overlay (post-EOS leakage)
# ==================================================================
fig_b, ax_b = plt.subplots(figsize=(11.5, 4.6), constrained_layout=True)

# 3 lines を別色で (3 本なので識別可、word ごとに区別)
colors = ["C0", "C1", "C2"]
for ci, p in enumerate(panels):
    ax_b.plot(range(77), p["dist_77"], color=colors[ci], lw=1.2, alpha=0.85,
              label=f"{p['word']!r} (q=({p['qy']},{p['qx']}))")

ymax = max(float(p["dist_77"].max()) for p in panels)
ax_b.set_ylim(0, ymax * 1.12)
ax_b.set_xlim(-0.5, 76.5)

ax_b.axvline(first_eos, color="red", lw=1.2, alpha=0.8, ls="--")
ax_b.text(first_eos - 0.4, ymax * 1.05, f"EOS (k={first_eos})",
          fontsize=9, color="red", ha="right", va="top")

ax_b.axvspan(first_eos + 0.5, 76.5, color="red", alpha=0.07, zorder=0)
ax_b.text((first_eos + 1 + 76) / 2, ymax * 0.55,
          f"padding 領域 (k={first_eos + 1}〜76, {76 - first_eos} 個)\n"
          "値は小さいが ≠ 0\n"
          "(softmax 必然 + EOS 系 hidden state の意味的延長)",
          fontsize=9, color="darkred", ha="center", va="center", alpha=0.9)

ax_b.set_xlabel("token position k (0=BOS, ..., 76=last padding EOS)", fontsize=9)
ax_b.set_ylabel(r"attention prob $\alpha_k$", fontsize=9)
ax_b.set_title(
    f"§7.11B {len(panels)}-panel overlay — EOS 以降への attention 漏れ "
    f"(first_eos={first_eos})",
    fontsize=10)
ax_b.grid(alpha=0.25)
ax_b.legend(loc="upper right", fontsize=9)

out_b = NB_OUT_DIR / "sec7_11b_post_eos_leakage.png"
fig_b.savefig(out_b, dpi=130)
print(f"saved: {out_b}")
plt.show()


**観察ポイント — Figure A: originating word peak の sanity + witch dominance / Figure B: post-EOS leakage の定量**

### 観察 1 (Figure A): 各 panel で originating word (緑枠) は確かに peak になる

3 panel すべてで、緑枠で囲んだ **originating word** が **その panel 内の最強 attention** (= 最も明るい strip cell) になっている — cross-attention の sanity check 通過:

- **`'witch'` panel** (q=(12, 36)): `witch` (k=3) が突出して明るく、他はほぼ暗い ← cleanest case
- **`'staff'` panel** (q=(9, 10)): `staff` (k=9) が緑枠で peak ← ただし `witch` (k=3) もほぼ同等に明るい
- **`'mushrooms'` panel** (q=(10, 3)): `mushrooms` (k=22) が緑枠で peak ← ただし `witch` (k=3) もほぼ同等に明るい

「query 位置がその word の cross-attn argmax だった」という選び方の必然として、その word が当該 query 行で最強になるのは想定通り。それでも他の strong peak が混じるかどうかは question of fact で、3 panel ともそうなっているのは sanity check として意味がある。

### 観察 2 (Figure A): **witch dominance** — `witch` (k=3) は 3 panel すべてで強い

staff / mushrooms panel でも `witch` (k=3) が peak とほぼ同レベルに明るい。この prompt では「画像内で最も視覚的に prominent な主語 token (= witch) が、別の word を query としても attention を吸う」現象が観察される。

**解釈案** (この prompt 1 例の範囲、断定不可):

- 主役 token は画像のあらゆる image 位置から見られている可能性 (画像中央の魔女が image 全体に semantic 影響を持つ)
- もしくは causal Transformer の前方 sink 効果 — BOS / `a` / 前方 content token への attention sink として一般に知られる現象と整合的

### 観察 3 (Figure B): post-EOS leakage は order ~10× 弱だが ≠ 0

EOS 境界 (k=32, 赤点線) より右側 = padding 領域 (k=33-76, 44 tokens):

- 3 線とも padding 領域で attention 値は **明確に 0 ではない** (≈ 0.005-0.01 程度)
- peak (witch panel の k=3 で ~0.26) に対し **~25-50× 弱い**
- 線形 plot では薄く見えるが、3 線とも padding 全域でほぼ一定の baseline 上で揺らいでいる

解釈は §7.4 cross-attn 注釈で既出: softmax の正規化制約 (negative を出せず必ず padding にも確率質量が分散) + EOS 系 hidden state が文末意味を保持し、padding token にも意味的延長として現れる。

→ 「padding は完全に無視されているわけではない」「CLIP encoder の EOS / 後続 padding は単なる文末マーカー以上の役割を持つ」という観察。実用上は無視可能 (peak の数 %) だが、構造的理解としては重要。

**観察範囲の制約 (caveats)**: §7.10 と共通の 3 panel / 1 step / head 平均 / t0 のみ / 3 query 位置のみ という狭い slice。**witch dominance** は単一 prompt × 単一 seed の anecdotal 現象、主語 token を別 word に置換した prompt で peak が移るかは本観察の範囲では確認していない。


<a id="cfg"></a>

---
## 8. guidance scale と negative prompt — 「prompt をどれくらい強く効かせるか」

CFG (Classifier-Free Guidance、§6.1 で出てきた) の `guidance_scale` `g` を変えて、生成画像が prompt にどれくらい強く引っ張られるかを比較します。さらに negative prompt をオン/オフして「負の方向の強制」が効くかを見ます。

**所要時間**: 1024×1024 / 30 steps × 4 値 + on/off × 2 = 6 枚生成 → Apple Silicon で **3〜5 分**。


In [ ]:
GUIDANCE_VALUES = [0.0, 3.0, 7.5, 12.0]

g_imgs: list[Image.Image] = []
g_meta: list[dict] = []
for g in GUIDANCE_VALUES:
    print(f"  generating g = {g} ...")
    gen3 = torch.Generator(device="cpu" if device == "mps" else device).manual_seed(seed)
    t0 = time.perf_counter()
    with torch.no_grad():
        out = pipe(
            prompt=prompt,
            negative_prompt=negative_prompt or None,
            width=width, height=height,
            num_inference_steps=num_inference_steps,
            guidance_scale=float(g),
            generator=gen3,
        )
    elapsed = time.perf_counter() - t0
    img = out.images[0]  # pyright: ignore[reportAttributeAccessIssue]
    g_imgs.append(img)
    g_meta.append({"g": float(g), "elapsed_sec": round(elapsed, 2)})
    print(f"    done in {elapsed:.1f} s")

fig, axes = plt.subplots(1, len(GUIDANCE_VALUES),
                         figsize=(len(GUIDANCE_VALUES) * 3.2, 3.6))
axes = np.atleast_1d(axes)
for ax, im, meta in zip(axes, g_imgs, g_meta):
    ax.imshow(im); ax.axis("off")
    ax.set_title(f"g = {meta['g']}", fontsize=10)
fig.suptitle(f"guidance scale sweep  ({width}x{height}, {num_inference_steps} steps)",
             fontsize=11)
fig.tight_layout()
out_path = NB_OUT_DIR / "sec8_guidance_grid.png"
fig.savefig(out_path, dpi=120)
print(f"saved: {out_path}")
plt.show()


In [ ]:
neg_results = []
for label, neg in [("no_negative", ""), ("with_negative", negative_prompt)]:
    print(f"  generating {label} ...")
    gen4 = torch.Generator(device="cpu" if device == "mps" else device).manual_seed(seed)
    t0 = time.perf_counter()
    with torch.no_grad():
        out = pipe(
            prompt=prompt,
            negative_prompt=neg or None,
            width=width, height=height,
            num_inference_steps=num_inference_steps,
            guidance_scale=guidance_scale,
            generator=gen4,
        )
    elapsed = time.perf_counter() - t0
    neg_results.append((label, out.images[0], elapsed))  # pyright: ignore[reportAttributeAccessIssue]
    print(f"    done in {elapsed:.1f} s")

fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.8))
for ax, (label, im, _) in zip(axes, neg_results):
    ax.imshow(im); ax.axis("off"); ax.set_title(label, fontsize=10)
fig.suptitle("negative prompt: off vs on (same prompt / seed / g)", fontsize=11)
fig.tight_layout()
out_path = NB_OUT_DIR / "sec8_negative_comparison.png"
fig.savefig(out_path, dpi=120)
print(f"saved: {out_path}")
plt.show()


**観察ポイント**

- **`g = 0`** は `guidance_scale <= 1` のため **CFG 外挿を使わない** モード (Diffusers の `do_classifier_free_guidance = guidance_scale > 1` のため): positive prompt のみで UNet を回し、`noise_pred = noise_pred_text`。**無条件生成ではなく**「prompt は効くが CFG 強化なし」状態。**negative_prompt も効かない** (uncond path 自体走らないため)。輪郭・彩度が控えめになりがち
- **`g = 3`** は穏やか。prompt の主要素は出るが彩度や detail が控えめ
- **`g = 7.5`** は **SD1.5 系で歴史的に使われてきた典型値** (Stability AI 公式 demo や AUTOMATIC1111/stable-diffusion-webui の歴史的 default)。**Diffusers の SDXL pipeline 既定値は 5.0**、本ノートでは比較対象として 7.5 を採用
- **`g = 12`** は過剰。色が飽和したり、prompt にない要素が削られ過ぎたりする (over-saturation, over-sharpening として知られる)
- **negative prompt** あり/なしは `blurry, low quality, distorted, ...` の embedding から **離れる方向** に押す効果 (CFG 有効時のみ作用)。[01ノートブック](01_sdxl_base_intro.ipynb) のような自然 prompt では大きな差は出にくいが、特に anatomy / quality 系で差が見えやすい

**`g` を上げれば良くなるわけではない**点が重要。「prompt の方向に強く押し過ぎると無理な対比が出る」というのは Classifier-Free Guidance の根本的なトレードオフです。


### CFG という名前について (Remark)

「Classifier-**Free** Guidance」(Ho & Salimans 2022, arXiv:2207.12598) という名前は、**先行する classifier guidance** (Dhariwal & Nichol 2021, arXiv:2105.05233) **という別 classifier を要する手法があり、それを classifier 不要にした**、という意味です。「Free = 分類器が不要」。命名の由来は「分類器（classifier）を別途要さない＝式に classifier 勾配が現れない」こと（"Classifier-Free"）です。なお CFG の合成式は Bayes 反転で得る implicit classifier の勾配に**着想を得た (inspired by)** 形ですが、原論文は「一般にはどの classifier の勾配でもない」と注意しており、この Bayes 等価は命名の由来ではなく効果の機構/動機です。

**Diffusers と原論文の `guidance_scale` 表記揺れ**: Diffusers (および AUTOMATIC1111/stable-diffusion-webui, ComfyUI など多くの実装) は §6.1 の式と同じ

```
ε̂ = ε_u + g · (ε_c − ε_u)
```

を使いますが、原論文 (HS2022) Eq. 6 は

```
ε̃ = (1 + w) · ε_c − w · ε_u
```

の形で書かれており、両者は

> **Diffusers の `guidance_scale`  =  論文の `w` + 1**

の関係。本ノートの `g = 7.5` は論文 `w = 6.5` に相当します。論文と実装で数字が違うときはこの 1 のオフセットを疑ってください。


<a id="vae"></a>

---
## 9. VAE roundtrip — 画像 ↔ latent ↔ 画像 で圧縮損失を見る

§6 で確認したように、SDXL Base は **VAE の latent 空間 (1, 4, 128, 128)** で diffusion を走らせ、最後だけ VAE decoder で pixel に戻します (1024×1024 → 128×128 → 1024×1024、空間 1 辺で **8 倍圧縮**)。

VAE 単体は学習済みの autoencoder なので、**画像 → latent → 画像 の roundtrip で「VAE が表現できる範囲」だけが残る**はずです。§6 で生成した final image を再 encode → decode して、どれくらいの劣化が出るかを確認します。


In [ ]:
from PIL import Image
import torchvision.transforms as T

# §6 final image を tensor 化 (1024×1024 RGB → (1, 3, H, W) range [-1, 1])
# 「final」= scheduler.step を最後まで適用した clean latent の decode 結果 = decoded[最終 step index]
final_img = decoded[num_inference_steps - 1]
to_tensor = T.Compose([
    T.ToTensor(),                          # [0, 1]
    T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),  # [-1, 1]
])

# winport fix(B-2 / 2026-06-29): この encode→decode roundtrip にも §6 の decode_latents_to_pil と
# 同じ fp16→fp32 局所 upcast を適用する。SDXL の fp16 VAE は encode が NaN にオーバーフローし
# （cuda/fp16 = Windows・Colab 共通。mps では不発火 = no-op）、decode 結果が真っ黒画像になるため。
# try/finally で元 dtype に戻す（恒久 upcast は後続セルの pipe() を壊す）。
_vae = pipe.vae
_orig_vae_dtype = next(_vae.parameters()).dtype
_upcast = _orig_vae_dtype == torch.float16
if _upcast:
    _vae.to(torch.float32)
try:
    _vae_dtype = next(_vae.parameters()).dtype
    # torchvision の Compose.__call__ は型注釈上 入力型を返す扱いになり、pyright が PIL.Image を推論する。
    # 実行時は ToTensor で Tensor に変換されるので unsqueeze で問題ないが、静的に解決できないので抑制。
    img_tensor = to_tensor(final_img).unsqueeze(0).to(device=device, dtype=_vae_dtype)  # pyright: ignore[reportAttributeAccessIssue]
    print("img_tensor shape :", tuple(img_tensor.shape))

    with torch.no_grad():
        # encode: VAE.encode は unscaled latent (DiagonalGaussianDistribution) を返す、
        # 呼び出し側で *scaling_factor して使う慣習
        posterior = _vae.encode(img_tensor).latent_dist
        latent_z = posterior.mean * _vae.config.scaling_factor
        print("latent (encoded) shape :", tuple(latent_z.shape),
              " mean/std :", f"{float(latent_z.mean()):+.3f} / {float(latent_z.std()):.3f}")
        print(f"vae.scaling_factor     : {_vae.config.scaling_factor} (SDXL は 0.13025、SD1.5 は 0.18215)")

        # decode (§6 の decode_latents_to_pil と同じ手順)
        z = latent_z / _vae.config.scaling_factor
        img_dec = _vae.decode(z, return_dict=False)[0]
finally:
    if _upcast:
        _vae.to(_orig_vae_dtype)

img_dec = (img_dec / 2 + 0.5).clamp(0, 1)
arr_dec = (img_dec[0].permute(1, 2, 0).float().cpu().numpy() * 255).astype(np.uint8)
roundtrip_img = Image.fromarray(arr_dec)

# 比較用 - L1 差分マップ
arr_orig = np.asarray(final_img).astype(np.float32)
arr_round = np.asarray(roundtrip_img).astype(np.float32)
diff = np.abs(arr_orig - arr_round).mean(axis=-1)
print()
print(f"roundtrip L1 diff: mean={diff.mean():.2f}  max={diff.max():.1f}  (0-255 range)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.6))
axes[0].imshow(final_img); axes[0].axis("off")
axes[0].set_title("§6 final image\n(diffusion output)", fontsize=10)
axes[1].imshow(roundtrip_img); axes[1].axis("off")
axes[1].set_title("VAE encode → decode roundtrip\n(same image: compressed → restored)", fontsize=10)
im = axes[2].imshow(diff, cmap="inferno", vmin=0, vmax=max(8.0, diff.max() * 0.5))
axes[2].axis("off")
axes[2].set_title(f"|diff| per pixel\nmean={diff.mean():.2f}  max={diff.max():.1f}",
                  fontsize=10)
fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
fig.suptitle("VAE roundtrip — 1024×1024 → 128×128 latent → 1024×1024", fontsize=11)
fig.tight_layout()
out_path = NB_OUT_DIR / "sec9_vae_roundtrip.png"
fig.savefig(out_path, dpi=120)
print(f"saved: {out_path}")
plt.show()


**観察ポイント**

- roundtrip 後も画像は **ほぼ同じに見える** (mean L1 < 5/255 程度のはず) — VAE がほとんど情報を失わずに「3×1024×1024 = 3 145 728 値」を「4×128×128 = 65 536 値」に押し込めている (圧縮比 約 48 倍)
- 差分マップを見ると、**細かいテクスチャ (髪の毛、葉の細部、文字や記号) で誤差が大きい**ことが分かる。VAE が苦手とするのは高周波領域 (細線、規則的パターン、テキスト)
- **`scaling_factor = 0.13025`** (SDXL は SD1.5 の 0.18215 より小さい) は再学習で latent 分散が変わったため。「encode 後に scaling_factor を掛けて、decode 前に割る」という慣習で diffusion を約 N(0, 1) スケールに揃えている ([01ノートブック §5](01_sdxl_base_intro.ipynb#inside) VAE 表参照)

VAE の「素直に再構成できる範囲」が、SDXL の画像表現の上限を決めています。diffusion がいくら頑張っても、最終出力は **VAE decoder が出せる画像にしかならない**。


<a id="summary"></a>

---
## まとめ — 画像生成 AI の中身を 5 ステップで

[01ノートブック](01_sdxl_base_intro.ipynb) が「pipeline をなめて読む」、02 が「**各段を手で分解して中身を見る**」でした。SDXL Base の forward を 5 段に再構成すると:

| ステップ | 入力 | 出力 | 主役モジュール | このノートでの確認 |
|---|---|---|---|---|
| ① tokenize | `prompt` (str) | 2 系統の token id 列 `(1, 77)` × 2 (同じ vocab) | `tokenizer` × 2 | §4.1 |
| ② text encode | token id 列 | `(1, 77, 768)` + `(1, 77, 1280)` + pooled `(1, 1280)` | `text_encoder` × 2 | §4.2, §4.3 |
| ③ condition pack | 上の 2 出力 | `prompt_embeds (1, 77, 2048)` + `pooled (1, 1280)` + `time_ids` | `pipe.encode_prompt()` | §4.4 |
| ④ denoise loop | 初期ノイズ `(1, 4, 128, 128)` + 上の condition | clean latent `(1, 4, 128, 128)` | `unet` + `scheduler` (30 step、CFG、cross-attn) | §6, §7, §8 |
| ⑤ decode | clean latent | 画像 `(3, 1024, 1024)` | `vae.decode` | §6, §9 |

**3 年前 SD1.x → SDXL Base で何が変わったか** ([01ノートブック](01_sdxl_base_intro.ipynb) まとめ + 今回の観察):

- ② text encode が **1 系統 → 2 系統 concat (768 + 1280 = 2048-d)**
- ③ pooled embedding + 解像度・crop 条件 を time embedding 経由で **追加注入** (`addition_embed_type="text_time"`)
- ④ UNet は **約 3 倍重く**、cross-attention の空間解像度は SD1.5 の 4 種から SDXL の **2 種 (64×64, 32×32)** に減ったが、同じ解像度に複数 transformer block が並ぶ (§7.1)
- ⑤ VAE は **構造同じ・重みは SDXL 用に再学習**、`scaling_factor` も 0.18215 → 0.13025 (§9)

### 次にどう繋がるか

- **prompt 探索系**: §7 の token table と attention 観察を発展させると、「同じ seed で prompt を少しずつ変えて出力差を見る」探索に繋がる
- **scheduler 系**: §6 を別 scheduler (DPM-Solver、DDIM など) に差し替えると、同じ条件で「scheduler の差で何が変わるか」が分かる
- **新世代モデル**: FLUX.1 / SD3.5 では UNet → MMDiT / DiT (Diffusion Transformer) に置き換わり、cross-attention の構造が大きく変わる。「中身を開けて見る」枠組みは同じで、attention の階層構造だけが違う

### 保存された出力

`outputs/02_sdxl_base_inside/` 配下:

- `sec5_embedding_geometry.png` — prompt embedding の PCA + t-SNE
- `sec6_trajectory_grid.png` + `sec6_step_*.png` (9 枚) + `sec6_final.png` + `sec6_latent_stats.{csv,png}` — denoising trajectory + latent stats
- `sec7_6_per_token_head0_{witch,mushrooms}.png` (2 枚) — §7.6 1 head raw attention
- `sec7_7_per_token_{witch,mushrooms}.png` (2 枚) — §7.7 .t0 head PCA-RGB
- `sec7_8_per_token_{witch,hair,staff,forest,mushrooms,magic_staff,enchanted_forest,bioluminescent_mushrooms}.png` (8 枚) — §7.8 全 t-block pool PCA-RGB
- `sec7_9_t0_profiles.png` — §7.9 profile-based head finding (interesting 10 t0 module を colored strip で stacked)
- `sec7_10_self_attn_grid.png` — §7.10 self-attention 3 panel (RANK_WORDS × top-1 head × cross-attn argmax 位置)
- `sec7_11a_token_strip.png` + `sec7_11b_post_eos_leakage.png` — §7.11 cross-attention 画像→単語 view (3 panel + post-EOS leakage)
- `sec8_guidance_grid.png` + `sec8_negative_comparison.png` — guidance sweep + negative on/off
- `sec9_vae_roundtrip.png` — VAE 圧縮損失

これらは notebook を再実行するたびに上書きされます。**同一環境** (同 PyTorch / Diffusers version + 同 device + 同 dtype + 同 attention processor) **+ 同じ seed + 同じ手順** であれば再現性が高く、出力 PNG はほぼ同じになります。**ただし bit-exact 同一は保証外** — version up / device 切り替え / fp 精度切り替え / processor 切り替えがあると微差が出ます。
